In [ ]:
"""
verificar_dataset.py
====================
Verificação obrigatória do dataset ANTES de qualquer treino.
Conforme Seção 7 do Contrato do Dataset GeoFM v2.

Execução:
    python verificar_dataset.py

O script imprime 10 amostras aleatórias e um checklist para cada uma.
Você deve ler cada amostra e confirmar manualmente os 5 critérios.
"""

import numpy as np
import random

# ============================================================
# CONFIGURAÇÃO — ajuste estes caminhos para o seu ambiente
# ============================================================
import os
import rasterio
from rasterio.windows import Window

DATA_DIR   = r"D:\Projetos\Cerrado\LULC"
PATTERN    = "brazil_coverage_{ano}_Cerrado.tif"
YEARS      = list(range(1985, 2025))   # 1985..2024

SOY_CODE   = 39
NODATA_IN  = 255
NODATA_OUT = 0     # valor que substitui nodata no array
PATCH_N    = 7     # tamanho do patch espacial (N×N)
HORIZON    = 5     # anos à frente para definir label (T+1 .. T+HORIZON)

# Anos válidos para T no conjunto de teste
T_RANGE_TESTE = list(range(2019, 2020))   # T=2019, label cobre 2020-2024

N_AMOSTRAS = 10
SEED       = 42

# ============================================================
# HELPERS
# ============================================================

def path(year):
    return os.path.join(DATA_DIR, PATTERN.format(ano=year))

def ler_pixel(year, row, col):
    """Lê um único valor de pixel de um raster."""
    with rasterio.open(path(year)) as ds:
        win = Window(col, row, 1, 1)
        v = ds.read(1, window=win, out_dtype="uint8")[0, 0]
    return int(v)

def ler_patch(year, row, col, n):
    """Lê um patch N×N centrado em (row, col)."""
    half = n // 2
    col0 = max(0, col - half)
    row0 = max(0, row - half)
    with rasterio.open(path(year)) as ds:
        W = ds.width
        H = ds.height
        col0 = min(col0, W - n)
        row0 = min(row0, H - n)
        win = Window(col0, row0, n, n)
        arr = ds.read(1, window=win, out_dtype="uint8")
    # substitui nodata
    arr = np.where(arr == NODATA_IN, NODATA_OUT, arr).astype(np.uint8)
    return arr

def serie_temporal(row, col, anos):
    """Retorna array 1D com a série temporal do pixel."""
    serie = []
    for y in anos:
        v = ler_pixel(y, row, col)
        v = NODATA_OUT if v == NODATA_IN else v
        serie.append(v)
    return np.array(serie, dtype=np.uint8)

def calcular_label(row, col, T, horizon):
    """
    Label = 1 se o pixel entra em SOY_CODE em algum ano de [T+1, T+horizon].
    Label = 0 caso contrário.
    """
    for y in range(T + 1, T + horizon + 1):
        if y > YEARS[-1]:
            break
        v = ler_pixel(y, row, col)
        if v == SOY_CODE:
            return 1, y  # retorna label e o ano em que virou soja
    return 0, None

def calcular_features_aux(serie):
    """Calcula features auxiliares a partir da série temporal."""
    has_21 = float(np.sum(serie == 21)) / len(serie)

    # anos consecutivos de pastagem (classe 15) antes do fim da série
    anos_pastagem = 0
    for v in reversed(serie):
        if v == 15:
            anos_pastagem += 1
        else:
            break

    classe_T_minus_1 = int(serie[-1])
    return has_21, anos_pastagem, classe_T_minus_1

# ============================================================
# SELEÇÃO DE AMOSTRAS ALEATÓRIAS
# ============================================================

def selecionar_amostras(n_amostras, seed):
    """
    Seleciona pixels aleatórios elegíveis:
    - Classe em T != SOY_CODE
    - Não dominados por nodata (< 50% da série)
    """
    random.seed(seed)
    np.random.seed(seed)

    with rasterio.open(path(YEARS[0])) as ds:
        H = ds.height
        W = ds.width

    amostras = []
    tentativas = 0
    max_tentativas = 5000

    print(f"\nSelecionando {n_amostras} amostras elegíveis (pode demorar alguns segundos)...\n")

    while len(amostras) < n_amostras and tentativas < max_tentativas:
        tentativas += 1
        T = random.choice(T_RANGE_TESTE)
        row = random.randint(PATCH_N // 2, H - PATCH_N // 2 - 1)
        col = random.randint(PATCH_N // 2, W - PATCH_N // 2 - 1)

        # classe em T
        classe_T = ler_pixel(T, row, col)
        if classe_T == SOY_CODE:
            continue   # não elegível

        # série até T-1
        anos_serie = [y for y in YEARS if y < T]
        serie = serie_temporal(row, col, anos_serie)

        # checar dominância de nodata
        pct_nodata = float(np.sum(serie == NODATA_OUT)) / len(serie)
        if pct_nodata > 0.5:
            continue   # dominado por nodata

        amostras.append({
            "T": T,
            "row": row,
            "col": col,
            "classe_T": classe_T,
            "anos_serie": anos_serie,
            "serie": serie,
        })

    if len(amostras) < n_amostras:
        print(f"AVISO: só foram encontradas {len(amostras)} amostras elegíveis em {tentativas} tentativas.")

    return amostras

# ============================================================
# VERIFICAÇÃO DE CADA AMOSTRA
# ============================================================

SEPARADOR = "=" * 70

def verificar_amostra(i, amostra):
    T      = amostra["T"]
    row    = amostra["row"]
    col    = amostra["col"]
    serie  = amostra["serie"]
    anos_s = amostra["anos_serie"]
    classe_T = amostra["classe_T"]

    print(SEPARADOR)
    print(f"AMOSTRA {i+1:02d} | Pixel: row={row}, col={col} | T={T}")
    print(SEPARADOR)

    # --- CRITÉRIO 1: série temporal termina em T-1 ---
    ultimo_ano_serie = anos_s[-1]
    criterio_1 = (ultimo_ano_serie == T - 1) and (T not in anos_s)
    print(f"\n[CRITÉRIO 1] Série termina em T-1={T-1}, sem dados de T em diante")
    print(f"  Primeiro ano da série : {anos_s[0]}")
    print(f"  Último ano da série   : {ultimo_ano_serie}")
    print(f"  T aparece na série?   : {T in anos_s}")
    print(f"  ✔ OK" if criterio_1 else "  ✘ FALHA — série inclui dados de T ou posterior!")

    # --- CRITÉRIO 2: série temporal (valores) ---
    print(f"\n[CRITÉRIO 2] Valores da série temporal (1985 a {T-1})")
    # imprime compactamente: ano: classe
    linha = ""
    for ano, v in zip(anos_s, serie):
        entrada = f"{ano}:{v:3d}  "
        if len(linha) + len(entrada) > 65:
            print("  " + linha)
            linha = ""
        linha += entrada
    if linha:
        print("  " + linha)
    pct_nodata = float(np.sum(serie == NODATA_OUT)) / len(serie) * 100
    print(f"  Nodata (0) na série   : {pct_nodata:.1f}%")
    criterio_2 = pct_nodata <= 50.0
    print(f"  ✔ OK (nodata <= 50%)" if criterio_2 else "  ✘ FALHA — nodata dominante!")

    # --- CRITÉRIO 3: pixel em T não é soja ---
    print(f"\n[CRITÉRIO 3] Pixel em T={T} NÃO é soja (classe {SOY_CODE})")
    print(f"  Classe em T={T}         : {classe_T}")
    criterio_3 = (classe_T != SOY_CODE)
    print(f"  ✔ OK" if criterio_3 else f"  ✘ FALHA — pixel já é soja em T! Não deveria estar no dataset.")

    # --- CRITÉRIO 4: label e verificação nos rasters ---
    label, ano_soja = calcular_label(row, col, T, HORIZON)
    print(f"\n[CRITÉRIO 4] Label e verificação nos rasters originais")
    print(f"  Horizonte verificado  : {T+1} a {min(T+HORIZON, YEARS[-1])}")
    if label == 1:
        print(f"  Label                 : 1 (pixel virou soja em {ano_soja})")
    else:
        print(f"  Label                 : 0 (pixel NÃO virou soja no horizonte)")
    # verificação cruzada: imprimir classes ano a ano no horizonte
    print(f"  Classes no horizonte  :")
    for y in range(T, min(T + HORIZON + 1, YEARS[-1] + 1)):
        v = ler_pixel(y, row, col)
        marcador = " <-- T" if y == T else (" <-- SOJA!" if v == SOY_CODE else "")
        print(f"    {y}: classe {v:3d}{marcador}")
    criterio_4 = True  # inspeção visual pelo usuário
    print(f"  ✔ Verifique visualmente se o label acima é coerente com as classes")

    # --- CRITÉRIO 5: patch espacial ---
    anos_patch = [y for y in YEARS if T - HORIZON <= y < T][-5:]  # últimos 5 anos antes de T
    print(f"\n[CRITÉRIO 5] Patch espacial {PATCH_N}×{PATCH_N} (últimos 5 anos antes de T)")
    print(f"  Anos do patch         : {anos_patch}")
    patches_ok = True
    for y in anos_patch:
        patch = ler_patch(y, row, col, PATCH_N)
        centro = patch[PATCH_N // 2, PATCH_N // 2]
        has_nodata_raw = np.any(patch == NODATA_IN)  # não deve ter: já convertido
        print(f"  {y}: shape={patch.shape}, centro={centro}, min={patch.min()}, max={patch.max()}, tem 255 residual={has_nodata_raw}")
        if has_nodata_raw:
            patches_ok = False
    criterio_5 = patches_ok
    print(f"  ✔ OK (sem nodata=255 residual)" if criterio_5 else "  ✘ FALHA — patch contém 255! Conversão de nodata não foi aplicada.")

    # --- FEATURES AUXILIARES ---
    has_21, anos_past, classe_Tm1 = calcular_features_aux(serie)
    print(f"\n[FEATURES AUXILIARES]")
    print(f"  has_21 (prop. anos c/ classe 21) : {has_21:.3f}")
    print(f"  anos_pastagem (consec. antes T)  : {anos_past}")
    print(f"  classe_T_minus_1                 : {classe_Tm1}")

    # --- RESUMO DO CHECKLIST ---
    print(f"\n{'─'*70}")
    print(f"CHECKLIST AMOSTRA {i+1:02d}:")
    print(f"  [{'✔' if criterio_1 else '✘'}] Série termina em T-1 sem dados futuros")
    print(f"  [{'✔' if criterio_2 else '✘'}] Nodata <= 50% da série")
    print(f"  [{'✔' if criterio_3 else '✘'}] Pixel em T não é soja")
    print(f"  [✔] Label verificável visualmente acima")
    print(f"  [{'✔' if criterio_5 else '✘'}] Patch sem nodata=255 residual")
    passou = criterio_1 and criterio_2 and criterio_3 and criterio_5
    print(f"\n  RESULTADO: {'✔ PASSOU' if passou else '✘ FALHA — investigar antes de continuar'}")
    print()

    return passou

# ============================================================
# MAIN
# ============================================================

if __name__ == "__main__":
    print("\n" + SEPARADOR)
    print("VERIFICAÇÃO DO DATASET — GeoFM v2")
    print("Conforme Seção 7 do Contrato do Dataset")
    print(SEPARADOR)
    print(f"Diretório de dados : {DATA_DIR}")
    print(f"Anos disponíveis   : {YEARS[0]}–{YEARS[-1]}")
    print(f"T de teste         : {T_RANGE_TESTE}")
    print(f"Horizonte de label : {HORIZON} anos")
    print(f"Tamanho do patch   : {PATCH_N}×{PATCH_N}")
    print(f"N de amostras      : {N_AMOSTRAS}")

    amostras = selecionar_amostras(N_AMOSTRAS, SEED)

    resultados = []
    for i, amostra in enumerate(amostras):
        passou = verificar_amostra(i, amostra)
        resultados.append(passou)

    # RESUMO FINAL
    print(SEPARADOR)
    print("RESUMO FINAL")
    print(SEPARADOR)
    n_passou = sum(resultados)
    print(f"Amostras verificadas : {len(resultados)}")
    print(f"Passou               : {n_passou}")
    print(f"Falhou               : {len(resultados) - n_passou}")
    print()
    if all(resultados):
        print("✔ TODAS AS AMOSTRAS PASSARAM.")
        print("  Você pode prosseguir para a implementação do dataset completo.")
    else:
        print("✘ EXISTEM FALHAS. NÃO prossiga para o treino.")
        print("  Investigue os critérios marcados com ✘ antes de continuar.")
    print()
    print("PRÓXIMO PASSO (após todas passarem):")
    print("  Implementar a classe Dataset (PyTorch) que retorna exatamente")
    print("  os campos verificados aqui, e rodar este script novamente")
    print("  sobre o __getitem__ do Dataset para confirmar consistência.")
    print(SEPARADOR)

In [1]:
"""
gerar_indice_raster.py
======================
Gera o índice de pixels para treino/validação/teste a partir do
raster classificado de origem da soja (soy2024_origin_classes).

Tarefa (revisada em relação ao contrato original):
  - Positivo (label=1): pixel com pastagem RÁPIDA antes de soja (classe 1, d<=4)
  - Negativo (label=0): pixel com pastagem LONGA antes de soja (classe 2, d>4)

Ambas as classes são trajetórias N->P->S confirmadas.
O modelo aprende a distinguir conversão rápida de lenta —
que é a pergunta central do projeto (estimação de k).

T para cada pixel = ano em que a pastagem começou após vegetação nativa.
O modelo vê a série 1985..T-1 e prediz se a conversão será rápida ou lenta.

Saída:
  indice_treino.csv  — T de 1990 a 2010
  indice_val.csv     — T de 2011 a 2015
  indice_teste.csv   — T de 2016 a 2019
"""

import os
import numpy as np
import pandas as pd
import rasterio
from rasterio.windows import Window
from collections import defaultdict

# ============================================================
# CONFIGURAÇÃO
# ============================================================
RASTER_CLASSES = r"D:\Projetos\Cerrado\LULC\soy2024_origin_classes_N4_P15_S39_k4_mask.tif"
DATA_DIR       = r"D:\Projetos\Cerrado\LULC"
PATTERN        = "brazil_coverage_{ano}_Cerrado.tif"
YEARS          = list(range(1985, 2025))
OUT_DIR        = r"D:\Projetos\Cerrado\GeoFM_sampling"

CLASSE_POS = 1   # pastagem rápida (d <= 4) → label = 1
CLASSE_NEG = 2   # pastagem longa  (d >  4) → label = 0

# Quantos pixels amostrar de cada classe por split
# (usa o mesmo N para positivos e negativos → dataset balanceado 50/50)
N_TREINO = 2000   # 1000 pos + 1000 neg
N_VAL    = 500    # 250  pos + 250  neg
N_TESTE  = 500    # 250  pos + 250  neg

BLOCK = 2048
SEED  = 42

# Split temporal por T (ano de início da pastagem)
# O modelo vê a série até T-1 e prediz rapidez da conversão
T_TREINO = list(range(1990, 2011))   # 21 anos
T_VAL    = list(range(2011, 2016))   # 5 anos
T_TESTE  = list(range(2016, 2020))   # 4 anos

SEP = "=" * 70

# ============================================================
# FUNÇÕES AUXILIARES
# ============================================================

def _path_lulc(year):
    return os.path.join(DATA_DIR, PATTERN.format(ano=year))

def _ler_pixel(year, row, col):
    with rasterio.open(_path_lulc(year)) as ds:
        v = ds.read(1, window=Window(col, row, 1, 1), out_dtype="uint8")[0, 0]
    return int(v)

def encontrar_ano_inicio_pastagem(row, col, years=YEARS):
    """
    Encontra o ano T em que o pixel entrou em pastagem (classe 15)
    após vegetação nativa (classe 4), varrendo a série temporal.
    Retorna T ou None se não encontrar padrão N->P.
    """
    prev = None
    for y in years:
        v = _ler_pixel(y, row, col)
        if prev == 4 and v == 15:   # transição N->P
            return y
        if v not in (0, 21):        # ignora nodata e mosaico
            prev = v
    return None

# ============================================================
# PASSO 1: VARRER O RASTER E COLETAR COORDENADAS POR CLASSE
# ============================================================
print(f"\n{SEP}")
print("PASSO 1: Coletando coordenadas por classe no raster classificado")
print(SEP)

coords_pos = []   # classe 1
coords_neg = []   # classe 2

with rasterio.open(RASTER_CLASSES) as ds:
    H, W = ds.height, ds.width
    print(f"Raster: {H} x {W} pixels")

    for row0 in range(0, H, BLOCK):
        bh = min(BLOCK, H - row0)
        for col0 in range(0, W, BLOCK):
            bw = min(BLOCK, W - col0)
            arr = ds.read(1, window=Window(col0, row0, bw, bh))

            # coordenadas absolutas dos pixels de interesse
            r_pos, c_pos = np.where(arr == CLASSE_POS)
            for r, c in zip(r_pos, c_pos):
                coords_pos.append((row0 + r, col0 + c))

            r_neg, c_neg = np.where(arr == CLASSE_NEG)
            for r, c in zip(r_neg, c_neg):
                coords_neg.append((row0 + r, col0 + c))

        if row0 % (BLOCK * 10) == 0:
            print(f"  Processado até linha {row0}/{H} | "
                  f"pos={len(coords_pos):,} neg={len(coords_neg):,}")

print(f"\nTotal coletado:")
print(f"  Positivos (classe 1): {len(coords_pos):,}")
print(f"  Negativos (classe 2): {len(coords_neg):,}")

# ============================================================
# PASSO 2: AMOSTRAR E ENCONTRAR ANO T
# ============================================================
print(f"\n{SEP}")
print("PASSO 2: Amostrando pixels e encontrando ano de início da pastagem (T)")
print(SEP)

rng = np.random.default_rng(SEED)

def amostrar_com_T(coords, n_total, t_range, label, desc):
    """
    Amostra até n_total pixels de coords, encontra T para cada um,
    e filtra pelos que têm T dentro de t_range.
    """
    # embaralha e pega uma amostra generosa (5x) para compensar filtros
    idx = rng.permutation(len(coords))[:min(len(coords), n_total * 5)]
    candidatos = [coords[i] for i in idx]

    registros = []
    print(f"\n  [{desc}] Buscando {n_total} pixels com T em {t_range[0]}..{t_range[-1]}...")

    for i, (row, col) in enumerate(candidatos):
        if len(registros) >= n_total:
            break

        T = encontrar_ano_inicio_pastagem(row, col)

        if T is None or T not in t_range:
            continue

        registros.append({
            "row"  : row,
            "col"  : col,
            "T"    : T,
            "label": label,
        })

        if len(registros) % 100 == 0:
            print(f"    {len(registros)}/{n_total} "
                  f"(verificados {i+1}/{len(candidatos)})")

    print(f"    Encontrados: {len(registros)}/{n_total}")
    return registros

# Treino
reg_treino_pos = amostrar_com_T(coords_pos, N_TREINO // 2, T_TREINO, 1, "TREINO positivos")
reg_treino_neg = amostrar_com_T(coords_neg, N_TREINO // 2, T_TREINO, 0, "TREINO negativos")

# Validação
reg_val_pos = amostrar_com_T(coords_pos, N_VAL // 2, T_VAL, 1, "VAL positivos")
reg_val_neg = amostrar_com_T(coords_neg, N_VAL // 2, T_VAL, 0, "VAL negativos")

# Teste
reg_teste_pos = amostrar_com_T(coords_pos, N_TESTE // 2, T_TESTE, 1, "TESTE positivos")
reg_teste_neg = amostrar_com_T(coords_neg, N_TESTE // 2, T_TESTE, 0, "TESTE negativos")

# ============================================================
# PASSO 3: MONTAR E SALVAR DATAFRAMES
# ============================================================
print(f"\n{SEP}")
print("PASSO 3: Salvando índices")
print(SEP)

os.makedirs(OUT_DIR, exist_ok=True)

def salvar(registros_pos, registros_neg, nome):
    df = pd.concat([
        pd.DataFrame(registros_pos),
        pd.DataFrame(registros_neg),
    ]).sample(frac=1, random_state=SEED).reset_index(drop=True)

    path = os.path.join(OUT_DIR, nome)
    df.to_csv(path, index=False)

    n_pos = df['label'].sum()
    n_neg = (df['label'] == 0).sum()
    print(f"\n  {nome}")
    print(f"    Total    : {len(df)}")
    print(f"    Positivos: {n_pos} ({100*n_pos/len(df):.1f}%)")
    print(f"    Negativos: {n_neg} ({100*n_neg/len(df):.1f}%)")
    print(f"    T range  : {df['T'].min()}–{df['T'].max()}")
    print(f"    Salvo em : {path}")
    return df

df_treino = salvar(reg_treino_pos, reg_treino_neg, "indice_treino.csv")
df_val    = salvar(reg_val_pos,    reg_val_neg,    "indice_val.csv")
df_teste  = salvar(reg_teste_pos,  reg_teste_neg,  "indice_teste.csv")

# ============================================================
# PASSO 4: VERIFICAÇÃO RÁPIDA
# ============================================================
print(f"\n{SEP}")
print("PASSO 4: Verificação cruzada (5 amostras aleatórias)")
print(SEP)
print("Confirmando que label no índice bate com classe no raster...\n")

sample = df_treino.sample(5, random_state=SEED)
with rasterio.open(RASTER_CLASSES) as ds:
    for _, r in sample.iterrows():
        v = ds.read(1, window=Window(int(r.col), int(r.row), 1, 1))[0, 0]
        esperado = CLASSE_POS if r.label == 1 else CLASSE_NEG
        ok = "✔" if int(v) == esperado else "✘"
        print(f"  {ok} row={int(r.row):6d} col={int(r.col):6d} "
              f"T={int(r.T)} label={int(r.label)} "
              f"classe_raster={int(v)} (esperado {esperado})")

print(f"\n{SEP}")
print("CONCLUÍDO")
print(SEP)
print("""
Próximos passos:
  1. Carregar indice_treino.csv / indice_val.csv / indice_teste.csv
     no LULCTransitionDataset (substitui o gerar_indice aleatório).
  2. Rodar verificar_dataset() sobre os novos índices.
  3. Rodar o baseline de regressão logística com os novos dados.
  4. Comparar balanced_accuracy e F1 — agora com positivos reais no teste.
""")



PASSO 1: Coletando coordenadas por classe no raster classificado
Raster: 82933 x 71227 pixels
  Processado até linha 0/82933 | pos=9 neg=1
  Processado até linha 20480/82933 | pos=63,847 neg=36,306
  Processado até linha 40960/82933 | pos=457,437 neg=286,262
  Processado até linha 61440/82933 | pos=529,353 neg=363,756
  Processado até linha 81920/82933 | pos=532,528 neg=373,339

Total coletado:
  Positivos (classe 1): 532,528
  Negativos (classe 2): 373,339

PASSO 2: Amostrando pixels e encontrando ano de início da pastagem (T)

  [TREINO positivos] Buscando 1000 pixels com T em 1990..2010...
    100/1000 (verificados 713/5000)
    200/1000 (verificados 1435/5000)
    300/1000 (verificados 2216/5000)
    400/1000 (verificados 3010/5000)
    500/1000 (verificados 3817/5000)
    600/1000 (verificados 4563/5000)
    Encontrados: 675/1000

  [TREINO negativos] Buscando 1000 pixels com T em 1990..2010...
    100/1000 (verificados 338/5000)
    200/1000 (verificados 635/5000)
    300/1000 (

TypeError: cannot convert the series to <class 'int'>

In [2]:
r"""
lulc_dataset.py
===============
Dataset PyTorch para predicao de transicao para soja no Cerrado.
Conforme Contrato do Dataset GeoFM v2.

Tarefa:
  - Positivo (label=1): pixel com pastagem RAPIDA antes de soja (d <= 4)
  - Negativo (label=0): pixel com pastagem LONGA antes de soja  (d >  4)

INPUT  : serie temporal do pixel (1985..T-1), padding a esquerda ate MAX_SERIE_LEN
         + serie_len (comprimento real)
         + patch espacial (5 anos x N x N), padding a esquerda se T proximo de 1985
         + features auxiliares (has_21, anos_pastagem, classe_T_minus_1)
OUTPUT : label binario 0 ou 1

USO:
  df = pd.read_csv(r"...\indice_treino.csv")
  ds = LULCTransitionDataset(df)
  verificar_dataset(ds, n=10)
"""

import os
import numpy as np
import pandas as pd
import rasterio
from rasterio.windows import Window
import torch
from torch.utils.data import Dataset

# ============================================================
# CONFIGURACAO GLOBAL
# ============================================================
DATA_DIR      = r"D:\Projetos\Cerrado\LULC"
PATTERN       = "brazil_coverage_{ano}_Cerrado.tif"
YEARS         = list(range(1985, 2025))
NODATA_IN     = 255
NODATA_OUT    = 0
PATCH_N       = 7
MAX_SERIE_LEN = len(YEARS) - 1   # = 39
PATCH_FRAMES  = 5                 # numero fixo de frames no patch

# ============================================================
# FUNCOES AUXILIARES
# ============================================================

def _path(year, data_dir=DATA_DIR, pattern=PATTERN):
    return os.path.join(data_dir, pattern.format(ano=year))

def _ler_pixel(year, row, col, data_dir=DATA_DIR, pattern=PATTERN):
    with rasterio.open(_path(year, data_dir, pattern)) as ds:
        v = ds.read(1, window=Window(col, row, 1, 1), out_dtype="uint8")[0, 0]
    return int(v)

def _ler_patch(year, row, col, n, data_dir=DATA_DIR, pattern=PATTERN):
    half = n // 2
    with rasterio.open(_path(year, data_dir, pattern)) as ds:
        H, W = ds.height, ds.width
        col0 = min(max(0, col - half), W - n)
        row0 = min(max(0, row - half), H - n)
        arr = ds.read(1, window=Window(col0, row0, n, n), out_dtype="uint8")
    return np.where(arr == NODATA_IN, NODATA_OUT, arr).astype(np.uint8)

# ============================================================
# DATASET PYTORCH
# ============================================================

class LULCTransitionDataset(Dataset):
    """
    Carrega amostras a partir de um indice CSV gerado por gerar_indice_raster.py.
    O CSV deve ter colunas: row, col, T, label.

    Retorna por amostra:
      serie     : float (MAX_SERIE_LEN,) -- serie 1985..T-1, zeros a esquerda
      serie_len : long  ()               -- comprimento real (sem padding)
      patch     : float (5, N, N)        -- ultimos 5 anos antes de T, zeros a esquerda
      patch_len : long  ()               -- frames reais no patch (sem padding)
      features  : float (3,)             -- has_21, anos_pastagem, classe_T_minus_1
      label     : long  ()               -- 0 ou 1
      meta      : dict  {row, col, T}    -- para debug
    """

    def __init__(
        self,
        indice_df,
        data_dir=DATA_DIR,
        pattern=PATTERN,
        years=YEARS,
        nodata_in=NODATA_IN,
        nodata_out=NODATA_OUT,
        patch_n=PATCH_N,
        max_serie_len=MAX_SERIE_LEN,
        patch_frames=PATCH_FRAMES,
    ):
        self.df            = indice_df.reset_index(drop=True)
        self.data_dir      = data_dir
        self.pattern       = pattern
        self.years         = years
        self.nodata_in     = nodata_in
        self.nodata_out    = nodata_out
        self.patch_n       = patch_n
        self.max_serie_len = max_serie_len
        self.patch_frames  = patch_frames

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        r     = self.df.iloc[idx]
        row   = int(r["row"])
        col   = int(r["col"])
        T     = int(r["T"])
        label = int(r["label"])

        # ── 1. Serie temporal (1985..T-1) com padding a esquerda ─────────
        anos_serie = [y for y in self.years if y < T]
        serie_raw  = np.array(
            [_ler_pixel(y, row, col, self.data_dir, self.pattern) for y in anos_serie],
            dtype=np.float32
        )
        serie_raw = np.where(
            serie_raw == self.nodata_in, self.nodata_out, serie_raw
        ).astype(np.float32)

        serie_len = len(serie_raw)
        serie     = np.zeros(self.max_serie_len, dtype=np.float32)
        serie[self.max_serie_len - serie_len:] = serie_raw   # alinha a direita

        # ── 2. Patch espacial (5 x N x N) com padding a esquerda ─────────
        anos_patch = [y for y in self.years if y < T][-self.patch_frames:]
        frames = [
            _ler_patch(y, row, col, self.patch_n, self.data_dir, self.pattern)
            for y in anos_patch
        ]

        # se houver menos de patch_frames anos disponiveis, preenche com zeros
        patch_len = len(frames)
        if patch_len < self.patch_frames:
            pad    = [np.zeros((self.patch_n, self.patch_n), dtype=np.uint8)] * (self.patch_frames - patch_len)
            frames = pad + frames   # zeros a esquerda

        patch = np.stack(frames, axis=0).astype(np.float32)  # sempre (5, N, N)

        # ── 3. Features auxiliares (sobre serie_raw, sem padding) ─────────
        has_21    = float(np.sum(serie_raw == 21)) / serie_len
        anos_past = 0
        for v in reversed(serie_raw):
            if v == 15:
                anos_past += 1
            else:
                break
        cl_tm1   = float(serie_raw[-1])
        features = np.array([has_21, float(anos_past), cl_tm1], dtype=np.float32)

        return {
            "serie"    : torch.tensor(serie,     dtype=torch.float32),
            "serie_len": torch.tensor(serie_len, dtype=torch.long),
            "patch"    : torch.tensor(patch,     dtype=torch.float32),
            "patch_len": torch.tensor(patch_len, dtype=torch.long),
            "features" : torch.tensor(features,  dtype=torch.float32),
            "label"    : torch.tensor(label,     dtype=torch.long),
            "meta"     : {"row": row, "col": col, "T": T},
        }

# ============================================================
# VERIFICACAO DO __getitem__
# ============================================================

def verificar_dataset(dataset, n=10, seed=42):
    """
    Verifica que o __getitem__ esta em conformidade com o contrato.
    Rodar sempre apos instanciar o dataset com um novo indice.
    """
    import random
    random.seed(seed)
    indices = random.sample(range(len(dataset)), min(n, len(dataset)))

    SEP = "=" * 70
    print(f"\n{SEP}\nVERIFICACAO DO __getitem__\n{SEP}")
    resultados = []

    for i, idx in enumerate(indices):
        s        = dataset[idx]
        r        = dataset.df.iloc[idx]
        T        = int(r["T"])
        row      = int(r["row"])
        col      = int(r["col"])
        lbl_esp  = int(r["label"])

        serie     = s["serie"].numpy()
        serie_len = int(s["serie_len"].item())
        patch     = s["patch"].numpy()
        patch_len = int(s["patch_len"].item())
        feats     = s["features"].numpy()
        label     = int(s["label"].item())

        anos_esp = [y for y in dataset.years if y < T]
        len_esp  = len(anos_esp)
        n_pad_s  = dataset.max_serie_len - serie_len
        n_pad_p  = dataset.patch_frames  - patch_len

        print(f"\nAMOSTRA {i+1:02d} | idx={idx} | row={row}, col={col} | T={T} | label={label}")
        print("─" * 70)

        # C1: serie
        c1 = (
            serie.shape == (dataset.max_serie_len,)
            and serie_len == len_esp
            and anos_esp[-1] == T - 1
            and (np.all(serie[:n_pad_s] == 0) if n_pad_s > 0 else True)
        )
        print(f"[C1] serie shape={serie.shape} | serie_len={serie_len} (esp {len_esp}) | "
              f"ultimo_ano={anos_esp[-1]} (esp {T-1}) | padding_ok={np.all(serie[:n_pad_s]==0) if n_pad_s>0 else True}")
        print(f"     {'OK' if c1 else 'FALHA'}")

        # C2: sem nodata=255
        c2 = not np.any(serie == 255)
        print(f"[C2] Sem nodata=255 na serie -> {'OK' if c2 else 'FALHA'}")

        # C3: patch shape e padding
        c3 = (
            patch.shape == (dataset.patch_frames, dataset.patch_n, dataset.patch_n)
            and not np.any(patch == 255)
            and (np.all(patch[:n_pad_p] == 0) if n_pad_p > 0 else True)
        )
        print(f"[C3] patch shape={patch.shape} | patch_len={patch_len} | "
              f"sem 255={not np.any(patch==255)} | padding_ok={np.all(patch[:n_pad_p]==0) if n_pad_p>0 else True}")
        print(f"     {'OK' if c3 else 'FALHA'}")

        # C4: label
        c4 = (label == lbl_esp)
        print(f"[C4] label={label} (esp {lbl_esp}) -> {'OK' if c4 else 'FALHA'}")

        # C5: features
        has_21, anos_past, cl_tm1 = feats
        c5 = (0.0 <= has_21 <= 1.0 and anos_past >= 0 and cl_tm1 >= 0)
        print(f"[C5] has_21={has_21:.3f} | anos_pastagem={int(anos_past)} | "
              f"classe_T-1={int(cl_tm1)} -> {'OK' if c5 else 'FALHA'}")

        passou = c1 and c2 and c3 and c4 and c5
        print(f"RESULTADO: {'PASSOU' if passou else 'FALHA'}")
        resultados.append(passou)

    print(f"\n{SEP}")
    print(f"RESUMO: {sum(resultados)}/{len(resultados)} passaram")
    if all(resultados):
        print("Dataset consistente com o contrato.")
        print("Proximo passo: rodar baseline_logreg.py")
    else:
        print("Existem falhas. Investigar antes de continuar.")
    print(SEP)
    return all(resultados)


# ============================================================
# EXECUCAO
# ============================================================
INDEX_DIR = r"D:\Projetos\Cerrado\GeoFM_sampling"

df_treino = pd.read_csv(os.path.join(INDEX_DIR, "indice_treino.csv"))
df_val    = pd.read_csv(os.path.join(INDEX_DIR, "indice_val.csv"))
df_teste  = pd.read_csv(os.path.join(INDEX_DIR, "indice_teste.csv"))

ds_treino = LULCTransitionDataset(df_treino)
ds_val    = LULCTransitionDataset(df_val)
ds_teste  = LULCTransitionDataset(df_teste)

print(f"Dataset treino : {len(ds_treino)} amostras")
print(f"Dataset val    : {len(ds_val)} amostras")
print(f"Dataset teste  : {len(ds_teste)} amostras")

verificar_dataset(ds_treino, n=10)

# Teste do DataLoader
from torch.utils.data import DataLoader
loader = DataLoader(ds_treino, batch_size=4, shuffle=True, num_workers=0)
batch  = next(iter(loader))
print(f"\nExemplo de batch:")
print(f"  serie     : {batch['serie'].shape}")
print(f"  serie_len : {batch['serie_len'].tolist()}")
print(f"  patch     : {batch['patch'].shape}")
print(f"  patch_len : {batch['patch_len'].tolist()}")
print(f"  features  : {batch['features'].shape}")
print(f"  label     : {batch['label'].tolist()}")

Dataset treino : 10000 amostras
Dataset val    : 1000 amostras
Dataset teste  : 420 amostras

VERIFICACAO DO __getitem__

AMOSTRA 01 | idx=1824 | row=38882, col=15060 | T=2008 | label=0
──────────────────────────────────────────────────────────────────────
[C1] serie shape=(39,) | serie_len=23 (esp 23) | ultimo_ano=2007 (esp 2007) | padding_ok=True
     OK
[C2] Sem nodata=255 na serie -> OK
[C3] patch shape=(5, 7, 7) | patch_len=5 | sem 255=True | padding_ok=True
     OK
[C4] label=0 (esp 0) -> OK
[C5] has_21=0.000 | anos_pastagem=0 | classe_T-1=4 -> OK
RESULTADO: PASSOU

AMOSTRA 02 | idx=409 | row=21933, col=50108 | T=2002 | label=1
──────────────────────────────────────────────────────────────────────
[C1] serie shape=(39,) | serie_len=17 (esp 17) | ultimo_ano=2001 (esp 2001) | padding_ok=True
     OK
[C2] Sem nodata=255 na serie -> OK
[C3] patch shape=(5, 7, 7) | patch_len=5 | sem 255=True | padding_ok=True
     OK
[C4] label=1 (esp 1) -> OK
[C5] has_21=0.000 | anos_pastagem=0 | cla

In [14]:
df_treino = pd.read_csv(r"D:\Projetos\Cerrado\GeoFM_sampling\indice_treino.csv")
ds_treino = LULCTransitionDataset(df_treino)
verificar_dataset(ds_treino, n=10)


VERIFICACAO DO __getitem__

AMOSTRA 01 | idx=1309 | row=36328, col=43997 | T=2013 | label=0
──────────────────────────────────────────────────────────────────────
[C1] serie shape=(39,) | serie_len=28 (esp 28) | ultimo_ano=2012 (esp 2012) | padding_ok=True
     OK
[C2] Sem nodata=255 na serie -> OK
[C3] patch shape=(5, 7, 7) | patch_len=5 | sem 255=True | padding_ok=True
     OK
[C4] label=0 (esp 0) -> OK
[C5] has_21=0.000 | anos_pastagem=0 | classe_T-1=4 -> OK
RESULTADO: PASSOU

AMOSTRA 02 | idx=228 | row=33099, col=45529 | T=2012 | label=0
──────────────────────────────────────────────────────────────────────
[C1] serie shape=(39,) | serie_len=27 (esp 27) | ultimo_ano=2011 (esp 2011) | padding_ok=True
     OK
[C2] Sem nodata=255 na serie -> OK
[C3] patch shape=(5, 7, 7) | patch_len=5 | sem 255=True | padding_ok=True
     OK
[C4] label=0 (esp 0) -> OK
[C5] has_21=0.000 | anos_pastagem=0 | classe_T-1=4 -> OK
RESULTADO: PASSOU

AMOSTRA 03 | idx=51 | row=16025, col=61618 | T=2020 | labe

True

In [16]:
"""
baseline_logreg.py
==================
Baseline de regressão logística para predição de conversão rápida vs lenta.
Conforme Contrato do Dataset GeoFM v2 — Seção 6 (Critério de sucesso mínimo).

Tarefa:
  - Positivo (label=1): pastagem rápida antes de soja (d <= 4)
  - Negativo (label=0): pastagem longa antes de soja  (d >  4)

Usa APENAS as 3 features auxiliares:
  - has_21          : proporção de anos com classe 21 na série
  - anos_pastagem   : anos consecutivos de pastagem antes de T
  - classe_T_minus_1: classe LULC do pixel em T-1

NÃO usa série temporal nem patch espacial.
O modelo DL precisa superar este baseline para justificar complexidade adicional.

PRÉ-REQUISITO: rodar gerar_indice_raster.py para gerar os CSVs de índice.
"""

import os
import json
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    balanced_accuracy_score, f1_score,
    classification_report, confusion_matrix,
)
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier

# ── dataset já definido na célula anterior do notebook ───────────────
# LULCTransitionDataset e verificar_dataset estão na memória do kernel

INDEX_DIR = r"D:\Projetos\Cerrado\GeoFM_sampling"
OUT_PATH  = os.path.join(INDEX_DIR, "baseline_results.json")

SEP = "=" * 70

# ============================================================
# 1. CARREGAR ÍNDICES
# ============================================================
print(f"\n{SEP}")
print("BASELINE — REGRESSÃO LOGÍSTICA")
print(SEP)

print("\n[1/4] Carregando índices...")

df_treino = pd.read_csv(os.path.join(INDEX_DIR, "indice_treino.csv"))
df_val    = pd.read_csv(os.path.join(INDEX_DIR, "indice_val.csv"))
df_teste  = pd.read_csv(os.path.join(INDEX_DIR, "indice_teste.csv"))

ds_treino = LULCTransitionDataset(df_treino)
ds_val    = LULCTransitionDataset(df_val)
ds_teste  = LULCTransitionDataset(df_teste)

for nome, df in [("Treino", df_treino), ("Val", df_val), ("Teste", df_teste)]:
    n_pos = df['label'].sum()
    print(f"  {nome:6s}: {len(df):5d} amostras | "
          f"pos={n_pos} ({100*n_pos/len(df):.1f}%) | "
          f"neg={(df['label']==0).sum()} ({100*(1-n_pos/len(df)):.1f}%) | "
          f"T={df['T'].min()}–{df['T'].max()}")
print(f"\nAVISO: teste tem {int(df_teste['label'].sum())} positivos "
      f"({100*df_teste['label'].mean():.1f}%) — assimetria geográfica documentada.")
# ============================================================
# 2. EXTRAIR FEATURES E LABELS
# ============================================================
print("\n[2/4] Extraindo features e labels...")

def extrair_X_y(dataset):
    loader = DataLoader(dataset, batch_size=64, shuffle=False, num_workers=0)
    X_list, y_list = [], []
    for batch in loader:
        X_list.append(batch["features"].numpy())
        y_list.append(batch["label"].numpy())
    return np.concatenate(X_list), np.concatenate(y_list)

X_tr, y_tr = extrair_X_y(ds_treino)
X_va, y_va = extrair_X_y(ds_val)
X_te, y_te = extrair_X_y(ds_teste)

print(f"  Treino : X={X_tr.shape} | pos={y_tr.sum()}")
print(f"  Val    : X={X_va.shape} | pos={y_va.sum()}")
print(f"  Teste  : X={X_te.shape} | pos={y_te.sum()}")
print(f"  Features: [has_21, anos_pastagem, classe_T_minus_1]")

# ============================================================
# 3. BASELINE 0 — PERSISTENCE
# ============================================================
print(f"\n{SEP}")
print("BASELINE 0 — PERSISTENCE (prediz sempre classe majoritária)")
print(SEP)

dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_tr, y_tr)
y_pred_dummy = dummy.predict(X_te)

bal_acc_dummy = balanced_accuracy_score(y_te, y_pred_dummy)
f1_dummy      = f1_score(y_te, y_pred_dummy, zero_division=0)
print(f"  Balanced accuracy : {bal_acc_dummy:.4f}")
print(f"  F1 (positivo)     : {f1_dummy:.4f}")

# ============================================================
# 4. BASELINE 1 — REGRESSÃO LOGÍSTICA
# ============================================================
print(f"\n{SEP}")
print("BASELINE 1 — REGRESSÃO LOGÍSTICA (3 features auxiliares)")
print(SEP)

scaler  = StandardScaler()
X_tr_sc = scaler.fit_transform(X_tr)
X_va_sc = scaler.transform(X_va)
X_te_sc = scaler.transform(X_te)

logreg = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)
logreg.fit(X_tr_sc, y_tr)

# Avaliação na validação
y_pred_va = logreg.predict(X_va_sc)
bal_acc_va = balanced_accuracy_score(y_va, y_pred_va)
f1_va      = f1_score(y_va, y_pred_va, zero_division=0)
print(f"  Validação — Balanced accuracy: {bal_acc_va:.4f} | F1: {f1_va:.4f}")

# Avaliação no teste
y_pred_lr = logreg.predict(X_te_sc)
bal_acc_lr = balanced_accuracy_score(y_te, y_pred_lr)
f1_lr      = f1_score(y_te, y_pred_lr, zero_division=0)
print(f"  Teste     — Balanced accuracy: {bal_acc_lr:.4f} | F1: {f1_lr:.4f}")

print(f"\n  Coeficientes (features normalizadas):")
feat_names = ["has_21", "anos_pastagem", "classe_T_minus_1"]
for name, coef in zip(feat_names, logreg.coef_[0]):
    print(f"    {name:25s}: {coef:+.4f}")

print(f"\n  Relatório de classificação (teste):")
print(classification_report(y_te, y_pred_lr,
                             target_names=["lenta/0", "rapida/1"],
                             zero_division=0))

cm = confusion_matrix(y_te, y_pred_lr)
print(f"  Matriz de confusão (linhas=real, colunas=predito):")
print(f"              pred_0  pred_1")
print(f"    real_0  :  {cm[0,0]:5d}   {cm[0,1]:5d}")
print(f"    real_1  :  {cm[1,0]:5d}   {cm[1,1]:5d}")

# ============================================================
# 5. RESUMO E CRITÉRIO DE SUCESSO
# ============================================================
print(f"\n{SEP}")
print("RESUMO — CRITÉRIO DE SUCESSO PARA O MODELO DL")
print(SEP)

print(f"\n  {'Baseline':<35} {'Bal. Accuracy':>15}  {'F1 (pos)':>10}")
print(f"  {'─'*35} {'─'*15}  {'─'*10}")
print(f"  {'Persistence':<35} {bal_acc_dummy:>15.4f}  {f1_dummy:>10.4f}")
print(f"  {'LogReg (3 features)':<35} {bal_acc_lr:>15.4f}  {f1_lr:>10.4f}")
print(f"\n  O modelo DL precisa superar {bal_acc_lr:.4f} em balanced accuracy")
print(f"  no conjunto de teste para justificar a série temporal e o patch.")

# ============================================================
# 6. SALVAR RESULTADOS
# ============================================================
results = {
    "descricao": "Baselines GeoFM v2 — Cerrado Pilot",
    "tarefa"   : "classificação rápida(1) vs lenta(0) — d<=4 vs d>4",
    "dados": {
        "treino": {"n": len(df_treino), "pos": int(df_treino['label'].sum()),
                   "T_range": f"{df_treino['T'].min()}-{df_treino['T'].max()}"},
        "val"   : {"n": len(df_val),    "pos": int(df_val['label'].sum()),
                   "T_range": f"{df_val['T'].min()}-{df_val['T'].max()}"},
        "teste" : {"n": len(df_teste),  "pos": int(df_teste['label'].sum()),
                   "T_range": f"{df_teste['T'].min()}-{df_teste['T'].max()}"},
    },
    "baselines": {
        "persistence": {
            "balanced_accuracy": round(float(bal_acc_dummy), 4),
            "f1_positivo"      : round(float(f1_dummy), 4),
        },
        "logreg_3features": {
            "balanced_accuracy_val" : round(float(bal_acc_va), 4),
            "balanced_accuracy_test": round(float(bal_acc_lr), 4),
            "f1_positivo_test"      : round(float(f1_lr), 4),
            "coeficientes": {n: round(float(c), 4)
                             for n, c in zip(feat_names, logreg.coef_[0])},
        },
    },
    "criterio_sucesso_dl": {
        "metrica_principal": "balanced_accuracy no teste",
        "valor_minimo"     : round(float(bal_acc_lr), 4),
        "nota": "Modelo DL deve superar logreg_3features para justificar complexidade"
    }
}

with open(OUT_PATH, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print(f"\n  Resultados salvos em: {OUT_PATH}")
print(f"\n{SEP}")
print("PRÓXIMO PASSO")
print(SEP)
print("""
  1. Anotar balanced_accuracy e F1 do baseline_results.json.
  2. Implementar modelo DL simples: MLP sobre features + média da série temporal.
  3. Comparar com estes números ANTES de aumentar complexidade.
  4. Só avançar para CNN/LSTM se o modelo simples já superar o baseline.
""")



BASELINE — REGRESSÃO LOGÍSTICA

[1/4] Carregando índices...
  Treino:  2000 amostras | pos=1000 (50.0%) | neg=1000 (50.0%) | T=1986–2020
  Val   :   500 amostras | pos=250 (50.0%) | neg=250 (50.0%) | T=1986–2019
  Teste :   420 amostras | pos=170 (40.5%) | neg=250 (59.5%) | T=1986–2022

AVISO: teste tem 170 positivos (40.5%) — assimetria geográfica documentada.

[2/4] Extraindo features e labels...
  Treino : X=(2000, 3) | pos=1000
  Val    : X=(500, 3) | pos=250
  Teste  : X=(420, 3) | pos=170
  Features: [has_21, anos_pastagem, classe_T_minus_1]

BASELINE 0 — PERSISTENCE (prediz sempre classe majoritária)
  Balanced accuracy : 0.5000
  F1 (positivo)     : 0.0000

BASELINE 1 — REGRESSÃO LOGÍSTICA (3 features auxiliares)
  Validação — Balanced accuracy: 0.3940 | F1: 0.5213
  Teste     — Balanced accuracy: 0.4072 | F1: 0.4881

  Coeficientes (features normalizadas):
    has_21                   : -0.0238
    anos_pastagem            : +0.0000
    classe_T_minus_1         : +0.0000

  R

In [7]:
"""
gerar_indice_raster.py
======================
Gera índice de pixels com SPLIT ESPACIAL por blocos geográficos.

Problema com split temporal:
  Classe 1 (rápida, d<=4) tem T recente por construção (conversão curta perto de 2024).
  Classe 2 (lenta,  d>4)  tem T antigo por construção (pastagem longa).
  Split por T separa as classes, não o tempo — invalida o experimento.

Solução — split espacial:
  Divide o raster em blocos de linhas (norte/centro/sul do Cerrado).
  Treino, val e teste recebem blocos diferentes — sem vazamento espacial.
  Cada bloco tem pixels de ambas as classes em proporção natural.

Split:
  Treino : 70% das linhas (blocos do norte/centro)
  Val    : 15% das linhas (bloco intermediário)
  Teste  : 15% das linhas (blocos do sul)
"""

import os
import numpy as np
import pandas as pd
import rasterio
from rasterio.windows import Window

# ============================================================
# CONFIGURAÇÃO
# ============================================================
RASTER_CLASSES = r"D:\Projetos\Cerrado\LULC\soy2024_origin_classes_N4_P15_S39_k4_mask.tif"
DATA_DIR       = r"D:\Projetos\Cerrado\LULC"
PATTERN        = "brazil_coverage_{ano}_Cerrado.tif"
YEARS          = list(range(1985, 2025))
OUT_DIR        = r"D:\Projetos\Cerrado\GeoFM_sampling"

CLASSE_POS = 1   # pastagem rápida (d <= 4) → label = 1
CLASSE_NEG = 2   # pastagem longa  (d >  4) → label = 0

# Amostras por split (positivos + negativos, balanceado 50/50)
N_TREINO = 2000   # 1000 pos + 1000 neg
N_VAL    =  500   #  250 pos +  250 neg
N_TESTE  =  500   #  250 pos +  250 neg

# Split espacial: frações de linhas do raster
FRAC_TREINO = 0.70
FRAC_VAL    = 0.15
# Teste = restante (0.15)

BLOCK = 2048
SEED  = 42

SEP = "=" * 70

# ============================================================
# FUNÇÕES AUXILIARES
# ============================================================

def _path_lulc(year):
    return os.path.join(DATA_DIR, PATTERN.format(ano=year))

def _ler_pixel(year, row, col):
    with rasterio.open(_path_lulc(year)) as ds:
        v = ds.read(1, window=Window(col, row, 1, 1), out_dtype="uint8")[0, 0]
    return int(v)

def encontrar_T(row, col):
    """
    Encontra o ano T em que o pixel entrou em pastagem (15)
    imediatamente após vegetação nativa (4).
    Retorna T ou None se não encontrar padrão N->P.
    """
    prev = None
    for y in YEARS:
        v = _ler_pixel(y, row, col)
        if v in (0, 21):
            prev = None   # quebra — reseta contexto
            continue
        if prev == 4 and v == 15:
            return y
        prev = v
    return None

# ============================================================
# PASSO 1: DEFINIR LIMITES DOS BLOCOS ESPACIAIS
# ============================================================
print(f"\n{SEP}")
print("PASSO 1: Definindo blocos espaciais de treino/val/teste")
print(SEP)

with rasterio.open(RASTER_CLASSES) as ds:
    H, W = ds.height, ds.width

row_treino_end = int(H * FRAC_TREINO)
row_val_end    = int(H * (FRAC_TREINO + FRAC_VAL))
row_teste_end  = H

print(f"Raster total   : {H} linhas x {W} colunas")
print(f"Treino (linhas): 0 a {row_treino_end-1}  ({row_treino_end} linhas, {FRAC_TREINO*100:.0f}%)")
print(f"Val    (linhas): {row_treino_end} a {row_val_end-1}  ({row_val_end-row_treino_end} linhas, {FRAC_VAL*100:.0f}%)")
print(f"Teste  (linhas): {row_val_end} a {row_teste_end-1}  ({row_teste_end-row_val_end} linhas, {(1-FRAC_TREINO-FRAC_VAL)*100:.0f}%)")

# ============================================================
# PASSO 2: COLETAR COORDENADAS POR CLASSE E POR BLOCO
# ============================================================
print(f"\n{SEP}")
print("PASSO 2: Coletando coordenadas por classe e bloco espacial")
print(SEP)

coords = {
    "treino": {CLASSE_POS: [], CLASSE_NEG: []},
    "val"   : {CLASSE_POS: [], CLASSE_NEG: []},
    "teste" : {CLASSE_POS: [], CLASSE_NEG: []},
}

def bloco_de(row):
    if row < row_treino_end: return "treino"
    if row < row_val_end:    return "val"
    return "teste"

with rasterio.open(RASTER_CLASSES) as ds:
    for row0 in range(0, H, BLOCK):
        bh = min(BLOCK, H - row0)
        for col0 in range(0, W, BLOCK):
            bw = min(BLOCK, W - col0)
            arr = ds.read(1, window=Window(col0, row0, bw, bh))

            for cls in (CLASSE_POS, CLASSE_NEG):
                rs, cs = np.where(arr == cls)
                for r, c in zip(rs, cs):
                    bloco = bloco_de(row0 + r)
                    coords[bloco][cls].append((row0 + r, col0 + c))

        if row0 % (BLOCK * 8) == 0:
            print(f"  Linha {row0:6d}/{H} | "
                  f"treino pos={len(coords['treino'][CLASSE_POS]):,} neg={len(coords['treino'][CLASSE_NEG]):,} | "
                  f"val pos={len(coords['val'][CLASSE_POS]):,} neg={len(coords['val'][CLASSE_NEG]):,} | "
                  f"teste pos={len(coords['teste'][CLASSE_POS]):,} neg={len(coords['teste'][CLASSE_NEG]):,}")

print(f"\nCoordenadas coletadas:")
for bloco in ("treino", "val", "teste"):
    print(f"  {bloco:6s}: pos={len(coords[bloco][CLASSE_POS]):,}  neg={len(coords[bloco][CLASSE_NEG]):,}")

# ============================================================
# PASSO 3: AMOSTRAR E ENCONTRAR T
# ============================================================
print(f"\n{SEP}")
print("PASSO 3: Amostrando pixels e encontrando T")
print(SEP)

rng = np.random.default_rng(SEED)

def amostrar(lista_coords, n, label, desc, max_factor=10):
    """
    Amostra n pixels de lista_coords, encontra T para cada um.
    max_factor: quantos candidatos verificar no máximo (n * max_factor).
    """
    idx       = rng.permutation(len(lista_coords))[:min(len(lista_coords), n * max_factor)]
    candidatos = [lista_coords[i] for i in idx]
    registros  = []

    print(f"\n  [{desc}] buscando {n} pixels...")
    for i, (row, col) in enumerate(candidatos):
        if len(registros) >= n:
            break
        T = encontrar_T(row, col)
        if T is None:
            continue
        registros.append({"row": row, "col": col, "T": T, "label": label})
        if len(registros) % 100 == 0:
            print(f"    {len(registros)}/{n} (verificados {i+1}/{len(candidatos)})")

    print(f"    Encontrados: {len(registros)}/{n}")
    return registros

splits_config = [
    ("treino", N_TREINO),
    ("val",    N_VAL),
    ("teste",  N_TESTE),
]

registros_por_bloco = {}
for bloco, n_total in splits_config:
    n_cada = n_total // 2
    pos = amostrar(coords[bloco][CLASSE_POS], n_cada, 1, f"{bloco} positivos")
    neg = amostrar(coords[bloco][CLASSE_NEG], n_cada, 0, f"{bloco} negativos")
    registros_por_bloco[bloco] = pos + neg

# ============================================================
# PASSO 4: SALVAR CSVs
# ============================================================
print(f"\n{SEP}")
print("PASSO 4: Salvando índices")
print(SEP)

os.makedirs(OUT_DIR, exist_ok=True)

dfs = {}
for bloco in ("treino", "val", "teste"):
    df = pd.DataFrame(registros_por_bloco[bloco])
    df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)

    path = os.path.join(OUT_DIR, f"indice_{bloco}.csv")
    df.to_csv(path, index=False)
    dfs[bloco] = df

    n_pos = int(df['label'].sum())
    n_neg = len(df) - n_pos
    print(f"\n  indice_{bloco}.csv")
    print(f"    Total    : {len(df)}")
    print(f"    Positivos: {n_pos} ({100*n_pos/len(df):.1f}%)")
    print(f"    Negativos: {n_neg} ({100*n_neg/len(df):.1f}%)")
    print(f"    T min/max: {df['T'].min()}–{df['T'].max()}")
    print(f"    Salvo em : {path}")

# ============================================================
# PASSO 5: VERIFICAÇÃO — T deve aparecer em ambas as classes
# ============================================================
print(f"\n{SEP}")
print("PASSO 5: Verificação de distribuição de T por classe")
print(SEP)
print("(T deve aparecer em ambas as classes em cada split)\n")

for bloco in ("treino", "val", "teste"):
    df = dfs[bloco]
    resumo = df.groupby('label')['T'].describe()[['min','50%','max']].rename(
        columns={'min':'T_min','50%':'T_med','max':'T_max'}
    )
    print(f"  {bloco}:")
    print(resumo.to_string())
    print()

# ============================================================
# PASSO 6: VERIFICAÇÃO CRUZADA COM O RASTER
# ============================================================
print(f"\n{SEP}")
print("PASSO 6: Verificação cruzada (5 amostras do teste)")
print(SEP)

sample = dfs["teste"].sample(5, random_state=SEED)
with rasterio.open(RASTER_CLASSES) as ds:
    for _, r in sample.iterrows():
        v = int(ds.read(1, window=Window(int(r["col"]), int(r["row"]), 1, 1))[0, 0])
        esperado = CLASSE_POS if r["label"] == 1 else CLASSE_NEG
        ok = "✔" if v == esperado else "✘"
        print(f"  {ok} row={int(r['row']):6d} col={int(r['col']):6d} "
              f"T={int(r['T'])} label={int(r['label'])} "
              f"classe_raster={v} (esperado {esperado})")

print(f"\n{SEP}")
print("CONCLUÍDO — próximos passos:")
print(SEP)
print("""
  1. Rodar lulc_dataset.py (carrega os novos CSVs e roda verificar_dataset)
  2. Rodar baseline_logreg.py (agora com teste balanceado)
  3. Observar se balanced_accuracy > 0.5 — as 3 features têm poder preditivo?
""")


PASSO 1: Definindo blocos espaciais de treino/val/teste
Raster total   : 82933 linhas x 71227 colunas
Treino (linhas): 0 a 58052  (58053 linhas, 70%)
Val    (linhas): 58053 a 70492  (12440 linhas, 15%)
Teste  (linhas): 70493 a 82932  (12440 linhas, 15%)

PASSO 2: Coletando coordenadas por classe e bloco espacial
  Linha      0/82933 | treino pos=9 neg=1 | val pos=0 neg=0 | teste pos=0 neg=0
  Linha  16384/82933 | treino pos=31,112 neg=15,694 | val pos=0 neg=0 | teste pos=0 neg=0
  Linha  32768/82933 | treino pos=350,780 neg=199,528 | val pos=0 neg=0 | teste pos=0 neg=0
  Linha  49152/82933 | treino pos=513,640 neg=345,162 | val pos=0 neg=0 | teste pos=0 neg=0
  Linha  65536/82933 | treino pos=527,553 neg=360,854 | val pos=4,621 neg=9,379 | teste pos=0 neg=0
  Linha  81920/82933 | treino pos=527,553 neg=360,854 | val pos=4,805 neg=10,018 | teste pos=170 neg=2,467

Coordenadas coletadas:
  treino: pos=527,553  neg=360,854
  val   : pos=4,805  neg=10,018
  teste : pos=170  neg=2,467

PAS

In [6]:
import pandas as pd
import os

INDEX_DIR = r"D:\Projetos\Cerrado\GeoFM_sampling"

for nome in ["indice_treino.csv", "indice_val.csv", "indice_teste.csv"]:
    df = pd.read_csv(os.path.join(INDEX_DIR, nome))
    print(f"\n{nome}")
    print(df.groupby(['label', 'T']).size().unstack(fill_value=0).to_string())


indice_treino.csv
T      1990  1991  1992  1993  1994  1995  1996  1997  1998  1999  2000  2001  2002  2003  2004  2005  2006  2007  2008  2009  2010
label                                                                                                                              
0        44    41    83    55    43    46    42    46    51    60    53    35    55    67    48    67    60    37    35    16    16
1        38    37    48    34    31    46    50    45    40    41    33    20    26    35    43    28    22    15    12    18    13

indice_val.csv
T      2011  2012  2013  2014  2015
label                              
0         5    80   106    44    15
1         0    35    73    87    55

indice_teste.csv
T      2016  2017  2018  2019
label                        
0         8     5     3     1
1       112    45    23     2


In [20]:
r"""
mlp_serie.py
============
Modelo DL minimo: MLP sobre a serie temporal achatada.
Primeira tentativa de superar o persistence baseline (bal_acc = 0.50).

Arquitetura deliberadamente simples:
  - Input: serie temporal (MAX_SERIE_LEN=39 valores) + 3 features auxiliares
  - 3 camadas lineares com ReLU e Dropout
  - Output: 2 classes (lenta/0, rapida/1)

Se este modelo nao superar 0.50 de balanced accuracy,
a serie temporal nao contem informacao util para a tarefa
-> investigar dados antes de aumentar complexidade.

Se superar -> justifica avancar para modelos sequenciais (LSTM, Conv1D).
"""

import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.metrics import balanced_accuracy_score, f1_score, confusion_matrix

# LULCTransitionDataset e verificar_dataset ja estao na memoria do kernel

INDEX_DIR = r"D:\Projetos\Cerrado\GeoFM_sampling"
SEP = "=" * 70

# ============================================================
# CONFIGURACAO
# ============================================================
BATCH_SIZE  = 32
LR          = 1e-3
EPOCHS      = 30
HIDDEN_DIMS = [128, 64, 32]
DROPOUT     = 0.3
SEED        = 42
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(SEED)
np.random.seed(SEED)

# ============================================================
# 1. CARREGAR DADOS
# ============================================================
print(f"\n{SEP}")
print("MLP SERIE TEMPORAL — CERRADO PILOT")
print(SEP)
print(f"Device: {DEVICE}")

df_treino = pd.read_csv(os.path.join(INDEX_DIR, "indice_treino.csv"))
df_val    = pd.read_csv(os.path.join(INDEX_DIR, "indice_val.csv"))
df_teste  = pd.read_csv(os.path.join(INDEX_DIR, "indice_teste.csv"))

ds_treino = LULCTransitionDataset(df_treino)
ds_val    = LULCTransitionDataset(df_val)
ds_teste  = LULCTransitionDataset(df_teste)

loader_treino = DataLoader(ds_treino, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
loader_val    = DataLoader(ds_val,    batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
loader_teste  = DataLoader(ds_teste,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"\nTreino: {len(ds_treino)} | Val: {len(ds_val)} | Teste: {len(ds_teste)}")

# ============================================================
# 2. MODELO
# ============================================================

class MLPSerie(nn.Module):
    """
    MLP que recebe:
      - serie temporal achatada (MAX_SERIE_LEN valores)
      - 3 features auxiliares
    Total de inputs: MAX_SERIE_LEN + 3 = 42
    """
    def __init__(self, serie_len=39, n_features=3, hidden_dims=HIDDEN_DIMS, dropout=DROPOUT):
        super().__init__()
        input_dim = serie_len + n_features

        layers = []
        prev   = input_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 2))   # 2 classes

        self.net = nn.Sequential(*layers)

    def forward(self, serie, features):
        # serie    : (B, 39)
        # features : (B, 3)
        x = torch.cat([serie, features], dim=1)   # (B, 42)
        return self.net(x)                         # (B, 2)


model = MLPSerie().to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())
print(f"\nModelo: MLPSerie | Parametros: {total_params:,}")
print(f"Arquitetura: 42 -> {' -> '.join(str(h) for h in HIDDEN_DIMS)} -> 2")

# ============================================================
# 3. TREINAMENTO
# ============================================================

# class_weight para compensar assimetria no teste (treino e 50/50 entao peso=1)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=5, verbose=False
)

def avaliar(loader):
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0.0
    with torch.no_grad():
        for batch in loader:
            serie    = batch["serie"].to(DEVICE)
            features = batch["features"].to(DEVICE)
            labels   = batch["label"].to(DEVICE)

            logits = model(serie, features)
            loss   = criterion(logits, labels)
            preds  = logits.argmax(dim=1)

            total_loss  += loss.item() * len(labels)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    bal_acc  = balanced_accuracy_score(all_labels, all_preds)
    f1       = f1_score(all_labels, all_preds, zero_division=0)
    return avg_loss, bal_acc, f1


print(f"\n{'Epoch':>6}  {'Loss Tr':>9}  {'BalAcc Tr':>10}  {'Loss Val':>9}  {'BalAcc Val':>11}  {'F1 Val':>7}")
print("─" * 65)

melhor_bal_acc_val = 0.0
melhor_epoch       = 0
melhor_state       = None
historico          = []

for epoch in range(1, EPOCHS + 1):
    # treino
    model.train()
    tr_loss  = 0.0
    tr_preds, tr_labels = [], []

    for batch in loader_treino:
        serie    = batch["serie"].to(DEVICE)
        features = batch["features"].to(DEVICE)
        labels   = batch["label"].to(DEVICE)

        optimizer.zero_grad()
        logits = model(serie, features)
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        preds = logits.argmax(dim=1)
        tr_loss += loss.item() * len(labels)
        tr_preds.extend(preds.cpu().numpy())
        tr_labels.extend(labels.cpu().numpy())

    tr_loss    /= len(loader_treino.dataset)
    tr_bal_acc  = balanced_accuracy_score(tr_labels, tr_preds)

    # validacao
    val_loss, val_bal_acc, val_f1 = avaliar(loader_val)
    scheduler.step(val_bal_acc)

    historico.append({
        "epoch": epoch, "tr_loss": tr_loss, "tr_bal_acc": tr_bal_acc,
        "val_loss": val_loss, "val_bal_acc": val_bal_acc, "val_f1": val_f1,
    })

    if val_bal_acc > melhor_bal_acc_val:
        melhor_bal_acc_val = val_bal_acc
        melhor_epoch       = epoch
        melhor_state       = {k: v.clone() for k, v in model.state_dict().items()}

    print(f"{epoch:>6}  {tr_loss:>9.4f}  {tr_bal_acc:>10.4f}  {val_loss:>9.4f}  {val_bal_acc:>11.4f}  {val_f1:>7.4f}")

print(f"\nMelhor validacao: epoch={melhor_epoch} | bal_acc={melhor_bal_acc_val:.4f}")

# ============================================================
# 4. AVALIACAO NO TESTE (melhor checkpoint)
# ============================================================
print(f"\n{SEP}")
print("AVALIACAO NO TESTE — melhor checkpoint")
print(SEP)

model.load_state_dict(melhor_state)
te_loss, te_bal_acc, te_f1 = avaliar(loader_teste)

# matriz de confusao
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for batch in loader_teste:
        logits = model(batch["serie"].to(DEVICE), batch["features"].to(DEVICE))
        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_labels.extend(batch["label"].numpy())

cm = confusion_matrix(all_labels, all_preds)
print(f"  Balanced accuracy : {te_bal_acc:.4f}")
print(f"  F1 (positivo)     : {te_f1:.4f}")
print(f"\n  Matriz de confusao (linhas=real, colunas=predito):")
print(f"              pred_0  pred_1")
print(f"    real_0  :  {cm[0,0]:5d}   {cm[0,1]:5d}")
print(f"    real_1  :  {cm[1,0]:5d}   {cm[1,1]:5d}")

# ============================================================
# 5. COMPARACAO COM BASELINES
# ============================================================
print(f"\n{SEP}")
print("COMPARACAO COM BASELINES")
print(SEP)

import json
baseline_path = os.path.join(INDEX_DIR, "baseline_results.json")
with open(baseline_path) as f:
    bl = json.load(f)

bl_persistence = bl["baselines"]["persistence"]["balanced_accuracy"]
bl_logreg      = bl["baselines"]["logreg_3features"]["balanced_accuracy_test"]

print(f"\n  {'Modelo':<35} {'Bal. Accuracy':>15}  {'F1 (pos)':>10}")
print(f"  {'─'*35} {'─'*15}  {'─'*10}")
print(f"  {'Persistence':<35} {bl_persistence:>15.4f}  {'—':>10}")
print(f"  {'LogReg (3 features)':<35} {bl_logreg:>15.4f}  {'—':>10}")
print(f"  {'MLP Serie (este modelo)':<35} {te_bal_acc:>15.4f}  {te_f1:>10.4f}")

superou = te_bal_acc > bl_persistence
print(f"\n  Superou persistence baseline (0.50)? {'SIM' if superou else 'NAO'}")

if superou:
    print("""
  A serie temporal contem informacao util para a tarefa.
  Proximo passo: substituir o MLP por um modelo sequencial (LSTM ou Conv1D)
  que leve em conta a ordem temporal dos valores.
    """)
else:
    print("""
  A serie temporal nao esta contribuindo acima do acaso.
  Antes de aumentar complexidade, investigar:
    1. A normalizacao dos valores de classe esta correta?
       (classes sao categorias, nao valores ordinais — considerar embedding)
    2. O numero de amostras e suficiente? (2000 pode ser pouco)
    3. Ha vazamento ou confusao nos labels?
  NAO avancar para LSTM/Conv antes de entender este resultado.
    """)

# salvar historico
hist_path = os.path.join(INDEX_DIR, "mlp_serie_historico.csv")
pd.DataFrame(historico).to_csv(hist_path, index=False)
print(f"  Historico de treino salvo em: {hist_path}")


MLP SERIE TEMPORAL — CERRADO PILOT
Device: cuda

Treino: 2000 | Val: 500 | Teste: 420

Modelo: MLPSerie | Parametros: 15,906
Arquitetura: 42 -> 128 -> 64 -> 32 -> 2

 Epoch    Loss Tr   BalAcc Tr   Loss Val   BalAcc Val   F1 Val
─────────────────────────────────────────────────────────────────


C:\Users\mario.barroso\AppData\Local\anaconda3\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


     1     0.6835      0.5680     0.5987       0.7700   0.7404
     2     0.6403      0.6525     0.5259       0.7880   0.7675
     3     0.6261      0.6675     0.5113       0.7940   0.7832
     4     0.6237      0.6640     0.5357       0.7680   0.7434
     5     0.6254      0.6705     0.5423       0.7660   0.7429
     6     0.6195      0.6725     0.5030       0.7860   0.7747
     7     0.6116      0.6725     0.5170       0.7900   0.7761
     8     0.6119      0.6755     0.5068       0.7880   0.7764
     9     0.6158      0.6825     0.5093       0.7900   0.7761
    10     0.6020      0.6755     0.5083       0.7920   0.7787
    11     0.6095      0.6790     0.5062       0.7780   0.7711
    12     0.6037      0.6780     0.5052       0.7960   0.7839
    13     0.6044      0.6815     0.4995       0.7940   0.7813
    14     0.6011      0.6775     0.4848       0.7920   0.7797
    15     0.5996      0.6780     0.4967       0.7940   0.7832
    16     0.5948      0.6790     0.4937       0.7960  

In [18]:
r"""
mlp_embedding.py
================
MLP com embedding de classes LULC.

Diferenca em relacao ao mlp_serie.py:
  - Os valores inteiros de classe (4, 15, 21, 39...) sao substituidos
    por vetores aprendidos via nn.Embedding
  - O modelo nao assume mais que classe 39 > classe 15 > classe 4
  - Cada classe recebe uma representacao densa de dimensao EMBED_DIM

Arquitetura:
  serie (39 valores inteiros) -> Embedding(128, EMBED_DIM) -> achatar
  concatenar com features auxiliares (3)
  -> MLP 3 camadas -> 2 classes

Se este modelo superar o MLP com valores brutos (bal_acc=0.5634),
o tratamento categorico das classes LULC e relevante.
"""

import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.metrics import balanced_accuracy_score, f1_score, confusion_matrix
import json

# LULCTransitionDataset ja esta na memoria do kernel

INDEX_DIR = r"D:\Projetos\Cerrado\GeoFM_sampling"
SEP = "=" * 70

# ============================================================
# CONFIGURACAO
# ============================================================
BATCH_SIZE  = 32
LR          = 1e-3
EPOCHS      = 30
EMBED_DIM   = 8       # dimensao do embedding por classe
HIDDEN_DIMS = [128, 64, 32]
DROPOUT     = 0.3
SEED        = 42
VOCAB_SIZE  = 128     # classes LULC sao uint8, max=127 (255 ja foi convertido para 0)
SERIE_LEN   = 39      # MAX_SERIE_LEN
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(SEED)
np.random.seed(SEED)

# ============================================================
# 1. CARREGAR DADOS
# ============================================================
print(f"\n{SEP}")
print("MLP COM EMBEDDING DE CLASSES LULC — CERRADO PILOT")
print(SEP)
print(f"Device    : {DEVICE}")
print(f"Embed dim : {EMBED_DIM} por classe | Vocab: {VOCAB_SIZE} classes")

df_treino = pd.read_csv(os.path.join(INDEX_DIR, "indice_treino.csv"))
df_val    = pd.read_csv(os.path.join(INDEX_DIR, "indice_val.csv"))
df_teste  = pd.read_csv(os.path.join(INDEX_DIR, "indice_teste.csv"))

ds_treino = LULCTransitionDataset(df_treino)
ds_val    = LULCTransitionDataset(df_val)
ds_teste  = LULCTransitionDataset(df_teste)

loader_treino = DataLoader(ds_treino, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
loader_val    = DataLoader(ds_val,    batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
loader_teste  = DataLoader(ds_teste,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Treino: {len(ds_treino)} | Val: {len(ds_val)} | Teste: {len(ds_teste)}")

# ============================================================
# 2. MODELO
# ============================================================

class MLPEmbedding(nn.Module):
    """
    MLP com embedding de classes LULC.

    Fluxo:
      serie (B, 39) inteiros
        -> Embedding -> (B, 39, EMBED_DIM)
        -> achatar   -> (B, 39 * EMBED_DIM)
        -> cat com features (B, 3)
        -> MLP -> (B, 2)
    """
    def __init__(
        self,
        vocab_size=VOCAB_SIZE,
        embed_dim=EMBED_DIM,
        serie_len=SERIE_LEN,
        n_features=3,
        hidden_dims=HIDDEN_DIMS,
        dropout=DROPOUT,
    ):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        # padding_idx=0 — o embedding do valor 0 (nodata/padding) fica fixo em zero

        input_dim = serie_len * embed_dim + n_features

        layers = []
        prev   = input_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 2))

        self.net = nn.Sequential(*layers)

    def forward(self, serie, features):
        # serie    : (B, 39) float -> converter para long para indexar embedding
        serie_int = serie.long()                      # (B, 39)
        emb       = self.embedding(serie_int)         # (B, 39, EMBED_DIM)
        emb_flat  = emb.view(emb.size(0), -1)         # (B, 39 * EMBED_DIM)
        x         = torch.cat([emb_flat, features], dim=1)
        return self.net(x)


model = MLPEmbedding().to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())
input_dim    = SERIE_LEN * EMBED_DIM + 3
print(f"\nModelo    : MLPEmbedding | Parametros: {total_params:,}")
print(f"Input dim : {SERIE_LEN} x {EMBED_DIM} (emb) + 3 (feat) = {input_dim}")
print(f"Arquitetura: {input_dim} -> {' -> '.join(str(h) for h in HIDDEN_DIMS)} -> 2")

# ============================================================
# 3. TREINAMENTO
# ============================================================

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=5
)

def avaliar(loader):
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0.0
    with torch.no_grad():
        for batch in loader:
            serie    = batch["serie"].to(DEVICE)
            features = batch["features"].to(DEVICE)
            labels   = batch["label"].to(DEVICE)
            logits   = model(serie, features)
            loss     = criterion(logits, labels)
            preds    = logits.argmax(dim=1)
            total_loss  += loss.item() * len(labels)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    avg_loss = total_loss / len(loader.dataset)
    bal_acc  = balanced_accuracy_score(all_labels, all_preds)
    f1       = f1_score(all_labels, all_preds, zero_division=0)
    return avg_loss, bal_acc, f1


print(f"\n{'Epoch':>6}  {'Loss Tr':>9}  {'BalAcc Tr':>10}  {'Loss Val':>9}  {'BalAcc Val':>11}  {'F1 Val':>7}")
print("─" * 65)

melhor_bal_acc_val = 0.0
melhor_epoch       = 0
melhor_state       = None
historico          = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    tr_loss = 0.0
    tr_preds, tr_labels = [], []

    for batch in loader_treino:
        serie    = batch["serie"].to(DEVICE)
        features = batch["features"].to(DEVICE)
        labels   = batch["label"].to(DEVICE)

        optimizer.zero_grad()
        logits = model(serie, features)
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        preds = logits.argmax(dim=1)
        tr_loss += loss.item() * len(labels)
        tr_preds.extend(preds.cpu().numpy())
        tr_labels.extend(labels.cpu().numpy())

    tr_loss   /= len(loader_treino.dataset)
    tr_bal_acc = balanced_accuracy_score(tr_labels, tr_preds)

    val_loss, val_bal_acc, val_f1 = avaliar(loader_val)
    scheduler.step(val_bal_acc)

    historico.append({
        "epoch": epoch, "tr_loss": tr_loss, "tr_bal_acc": tr_bal_acc,
        "val_loss": val_loss, "val_bal_acc": val_bal_acc, "val_f1": val_f1,
    })

    if val_bal_acc > melhor_bal_acc_val:
        melhor_bal_acc_val = val_bal_acc
        melhor_epoch       = epoch
        melhor_state       = {k: v.clone() for k, v in model.state_dict().items()}

    print(f"{epoch:>6}  {tr_loss:>9.4f}  {tr_bal_acc:>10.4f}  {val_loss:>9.4f}  {val_bal_acc:>11.4f}  {val_f1:>7.4f}")

print(f"\nMelhor validacao: epoch={melhor_epoch} | bal_acc={melhor_bal_acc_val:.4f}")

# ============================================================
# 4. AVALIACAO NO TESTE
# ============================================================
print(f"\n{SEP}")
print("AVALIACAO NO TESTE — melhor checkpoint")
print(SEP)

model.load_state_dict(melhor_state)
te_loss, te_bal_acc, te_f1 = avaliar(loader_teste)

model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for batch in loader_teste:
        logits = model(batch["serie"].to(DEVICE), batch["features"].to(DEVICE))
        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_labels.extend(batch["label"].numpy())

cm = confusion_matrix(all_labels, all_preds)
print(f"  Balanced accuracy : {te_bal_acc:.4f}")
print(f"  F1 (positivo)     : {te_f1:.4f}")
print(f"\n  Matriz de confusao:")
print(f"              pred_0  pred_1")
print(f"    real_0  :  {cm[0,0]:5d}   {cm[0,1]:5d}")
print(f"    real_1  :  {cm[1,0]:5d}   {cm[1,1]:5d}")

# ============================================================
# 5. COMPARACAO COMPLETA
# ============================================================
print(f"\n{SEP}")
print("COMPARACAO COMPLETA")
print(SEP)

with open(os.path.join(INDEX_DIR, "baseline_results.json")) as f:
    bl = json.load(f)

bl_persistence = bl["baselines"]["persistence"]["balanced_accuracy"]
bl_logreg      = bl["baselines"]["logreg_3features"]["balanced_accuracy_test"]
mlp_serie_acc  = 0.5634   # resultado anterior

print(f"\n  {'Modelo':<40} {'Bal. Acc (teste)':>16}  {'F1 (pos)':>10}")
print(f"  {'─'*40} {'─'*16}  {'─'*10}")
print(f"  {'Persistence':<40} {bl_persistence:>16.4f}  {'—':>10}")
print(f"  {'LogReg (3 features)':<40} {bl_logreg:>16.4f}  {'—':>10}")
print(f"  {'MLP serie (valores brutos)':<40} {mlp_serie_acc:>16.4f}  {'0.2634':>10}")
print(f"  {'MLP embedding (este modelo)':<40} {te_bal_acc:>16.4f}  {te_f1:>10.4f}")

delta = te_bal_acc - mlp_serie_acc
print(f"\n  Ganho do embedding sobre valores brutos: {delta:+.4f}")

if te_bal_acc > mlp_serie_acc:
    print("""
  O tratamento categorico das classes LULC melhora o modelo.
  O embedding aprendeu representacoes uteis para cada classe.
  Proximo passo: LSTM ou Conv1D para capturar ordem temporal.
    """)
else:
    print("""
  O embedding nao trouxe ganho sobre valores brutos nesta configuracao.
  Possivel causa: com apenas 39 passos e MLP achatado, a ordem nao importa --
  o modelo nao consegue usar a estrutura sequencial de qualquer forma.
  Proximo passo: LSTM ou Conv1D — a ordem temporal pode ser o que falta.
    """)

# salvar historico
pd.DataFrame(historico).to_csv(
    os.path.join(INDEX_DIR, "mlp_embedding_historico.csv"), index=False
)
print(f"  Historico salvo em: mlp_embedding_historico.csv")


MLP COM EMBEDDING DE CLASSES LULC — CERRADO PILOT
Device    : cuda
Embed dim : 8 por classe | Vocab: 128 classes
Treino: 2000 | Val: 500 | Teste: 420

Modelo    : MLPEmbedding | Parametros: 51,874
Input dim : 39 x 8 (emb) + 3 (feat) = 315
Arquitetura: 315 -> 128 -> 64 -> 32 -> 2

 Epoch    Loss Tr   BalAcc Tr   Loss Val   BalAcc Val   F1 Val
─────────────────────────────────────────────────────────────────
     1     0.6850      0.5585     0.6550       0.6420   0.4900
     2     0.6535      0.6265     0.5634       0.8040   0.7958
     3     0.6354      0.6465     0.5153       0.8180   0.8084
     4     0.6180      0.6635     0.5073       0.7900   0.7722
     5     0.6122      0.6675     0.5381       0.8000   0.7890
     6     0.6073      0.6710     0.5188       0.8000   0.7951
     7     0.5996      0.6805     0.5088       0.7960   0.7927
     8     0.6010      0.6685     0.5181       0.8120   0.8017
     9     0.5931      0.6880     0.5177       0.7940   0.7911
    10     0.5905     

In [21]:
r"""
mlp_embedding.py
================
MLP com embedding de classes LULC.

Diferenca em relacao ao mlp_serie.py:
  - Os valores inteiros de classe (4, 15, 21, 39...) sao substituidos
    por vetores aprendidos via nn.Embedding
  - O modelo nao assume mais que classe 39 > classe 15 > classe 4
  - Cada classe recebe uma representacao densa de dimensao EMBED_DIM

Arquitetura:
  serie (39 valores inteiros) -> Embedding(128, EMBED_DIM) -> achatar
  concatenar com features auxiliares (3)
  -> MLP 3 camadas -> 2 classes

Se este modelo superar o MLP com valores brutos (bal_acc=0.5634),
o tratamento categorico das classes LULC e relevante.
"""

import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.metrics import balanced_accuracy_score, f1_score, confusion_matrix
import json

# LULCTransitionDataset ja esta na memoria do kernel

INDEX_DIR = r"D:\Projetos\Cerrado\GeoFM_sampling"
SEP = "=" * 70

# ============================================================
# CONFIGURACAO
# ============================================================
BATCH_SIZE  = 32
LR          = 1e-3
EPOCHS      = 30
EMBED_DIM   = 8       # dimensao do embedding por classe
HIDDEN_DIMS = [128, 64, 32]
DROPOUT     = 0.3
SEED        = 42
VOCAB_SIZE  = 128     # classes LULC sao uint8, max=127 (255 ja foi convertido para 0)
SERIE_LEN   = 39      # MAX_SERIE_LEN
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(SEED)
np.random.seed(SEED)

# ============================================================
# 1. CARREGAR DADOS
# ============================================================
print(f"\n{SEP}")
print("MLP COM EMBEDDING DE CLASSES LULC — CERRADO PILOT")
print(SEP)
print(f"Device    : {DEVICE}")
print(f"Embed dim : {EMBED_DIM} por classe | Vocab: {VOCAB_SIZE} classes")

df_treino = pd.read_csv(os.path.join(INDEX_DIR, "indice_treino.csv"))
df_val    = pd.read_csv(os.path.join(INDEX_DIR, "indice_val.csv"))
df_teste  = pd.read_csv(os.path.join(INDEX_DIR, "indice_teste.csv"))

ds_treino = LULCTransitionDataset(df_treino)
ds_val    = LULCTransitionDataset(df_val)
ds_teste  = LULCTransitionDataset(df_teste)

loader_treino = DataLoader(ds_treino, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
loader_val    = DataLoader(ds_val,    batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
loader_teste  = DataLoader(ds_teste,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Treino: {len(ds_treino)} | Val: {len(ds_val)} | Teste: {len(ds_teste)}")

# ============================================================
# 2. MODELO
# ============================================================

class MLPEmbedding(nn.Module):
    """
    MLP com embedding de classes LULC.

    Fluxo:
      serie (B, 39) inteiros
        -> Embedding -> (B, 39, EMBED_DIM)
        -> achatar   -> (B, 39 * EMBED_DIM)
        -> cat com features (B, 3)
        -> MLP -> (B, 2)
    """
    def __init__(
        self,
        vocab_size=VOCAB_SIZE,
        embed_dim=EMBED_DIM,
        serie_len=SERIE_LEN,
        n_features=3,
        hidden_dims=HIDDEN_DIMS,
        dropout=DROPOUT,
    ):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        # padding_idx=0 — o embedding do valor 0 (nodata/padding) fica fixo em zero

        input_dim = serie_len * embed_dim + n_features

        layers = []
        prev   = input_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 2))

        self.net = nn.Sequential(*layers)

    def forward(self, serie, features):
        # serie    : (B, 39) float -> converter para long para indexar embedding
        serie_int = serie.long()                      # (B, 39)
        emb       = self.embedding(serie_int)         # (B, 39, EMBED_DIM)
        emb_flat  = emb.view(emb.size(0), -1)         # (B, 39 * EMBED_DIM)
        x         = torch.cat([emb_flat, features], dim=1)
        return self.net(x)


model = MLPEmbedding().to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())
input_dim    = SERIE_LEN * EMBED_DIM + 3
print(f"\nModelo    : MLPEmbedding | Parametros: {total_params:,}")
print(f"Input dim : {SERIE_LEN} x {EMBED_DIM} (emb) + 3 (feat) = {input_dim}")
print(f"Arquitetura: {input_dim} -> {' -> '.join(str(h) for h in HIDDEN_DIMS)} -> 2")

# ============================================================
# 3. TREINAMENTO
# ============================================================

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=5
)

def avaliar(loader):
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0.0
    with torch.no_grad():
        for batch in loader:
            serie    = batch["serie"].to(DEVICE)
            features = batch["features"].to(DEVICE)
            labels   = batch["label"].to(DEVICE)
            logits   = model(serie, features)
            loss     = criterion(logits, labels)
            preds    = logits.argmax(dim=1)
            total_loss  += loss.item() * len(labels)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    avg_loss = total_loss / len(loader.dataset)
    bal_acc  = balanced_accuracy_score(all_labels, all_preds)
    f1       = f1_score(all_labels, all_preds, zero_division=0)
    return avg_loss, bal_acc, f1


print(f"\n{'Epoch':>6}  {'Loss Tr':>9}  {'BalAcc Tr':>10}  {'Loss Val':>9}  {'BalAcc Val':>11}  {'F1 Val':>7}")
print("─" * 65)

melhor_bal_acc_val = 0.0
melhor_epoch       = 0
melhor_state       = None
historico          = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    tr_loss = 0.0
    tr_preds, tr_labels = [], []

    for batch in loader_treino:
        serie    = batch["serie"].to(DEVICE)
        features = batch["features"].to(DEVICE)
        labels   = batch["label"].to(DEVICE)

        optimizer.zero_grad()
        logits = model(serie, features)
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        preds = logits.argmax(dim=1)
        tr_loss += loss.item() * len(labels)
        tr_preds.extend(preds.cpu().numpy())
        tr_labels.extend(labels.cpu().numpy())

    tr_loss   /= len(loader_treino.dataset)
    tr_bal_acc = balanced_accuracy_score(tr_labels, tr_preds)

    val_loss, val_bal_acc, val_f1 = avaliar(loader_val)
    scheduler.step(val_bal_acc)

    historico.append({
        "epoch": epoch, "tr_loss": tr_loss, "tr_bal_acc": tr_bal_acc,
        "val_loss": val_loss, "val_bal_acc": val_bal_acc, "val_f1": val_f1,
    })

    if val_bal_acc > melhor_bal_acc_val:
        melhor_bal_acc_val = val_bal_acc
        melhor_epoch       = epoch
        melhor_state       = {k: v.clone() for k, v in model.state_dict().items()}

    print(f"{epoch:>6}  {tr_loss:>9.4f}  {tr_bal_acc:>10.4f}  {val_loss:>9.4f}  {val_bal_acc:>11.4f}  {val_f1:>7.4f}")

print(f"\nMelhor validacao: epoch={melhor_epoch} | bal_acc={melhor_bal_acc_val:.4f}")

# ============================================================
# 4. AVALIACAO NO TESTE
# ============================================================
print(f"\n{SEP}")
print("AVALIACAO NO TESTE — melhor checkpoint")
print(SEP)

model.load_state_dict(melhor_state)
te_loss, te_bal_acc, te_f1 = avaliar(loader_teste)

model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for batch in loader_teste:
        logits = model(batch["serie"].to(DEVICE), batch["features"].to(DEVICE))
        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_labels.extend(batch["label"].numpy())

cm = confusion_matrix(all_labels, all_preds)
print(f"  Balanced accuracy : {te_bal_acc:.4f}")  
print(f"  F1 (positivo)     : {te_f1:.4f}")
print(f"\n  Matriz de confusao:")
print(f"              pred_0  pred_1")
print(f"    real_0  :  {cm[0,0]:5d}   {cm[0,1]:5d}")
print(f"    real_1  :  {cm[1,0]:5d}   {cm[1,1]:5d}")

# ============================================================
# 5. COMPARACAO COMPLETA
# ============================================================
print(f"\n{SEP}")
print("COMPARACAO COMPLETA")
print(SEP)

with open(os.path.join(INDEX_DIR, "baseline_results.json")) as f:
    bl = json.load(f)

bl_persistence = bl["baselines"]["persistence"]["balanced_accuracy"]
bl_logreg      = bl["baselines"]["logreg_3features"]["balanced_accuracy_test"]
mlp_serie_acc  = 0.5634   # resultado anterior

print(f"\n  {'Modelo':<40} {'Bal. Acc (teste)':>16}  {'F1 (pos)':>10}")
print(f"  {'─'*40} {'─'*16}  {'─'*10}")
print(f"  {'Persistence':<40} {bl_persistence:>16.4f}  {'—':>10}")
print(f"  {'LogReg (3 features)':<40} {bl_logreg:>16.4f}  {'—':>10}")
print(f"  {'MLP serie (valores brutos)':<40} {mlp_serie_acc:>16.4f}  {'0.2634':>10}")
print(f"  {'MLP embedding (este modelo)':<40} {te_bal_acc:>16.4f}  {te_f1:>10.4f}")

delta = te_bal_acc - mlp_serie_acc
print(f"\n  Ganho do embedding sobre valores brutos: {delta:+.4f}")

if te_bal_acc > mlp_serie_acc:
    print("""
  O tratamento categorico das classes LULC melhora o modelo.
  O embedding aprendeu representacoes uteis para cada classe.
  Proximo passo: LSTM ou Conv1D para capturar ordem temporal.
    """)
else:
    print("""
  O embedding nao trouxe ganho sobre valores brutos nesta configuracao.
  Possivel causa: com apenas 39 passos e MLP achatado, a ordem nao importa --
  o modelo nao consegue usar a estrutura sequencial de qualquer forma.
  Proximo passo: LSTM ou Conv1D — a ordem temporal pode ser o que falta.
    """)

# salvar historico
pd.DataFrame(historico).to_csv(
    os.path.join(INDEX_DIR, "mlp_embedding_historico.csv"), index=False
)
print(f"  Historico salvo em: mlp_embedding_historico.csv")


MLP COM EMBEDDING DE CLASSES LULC — CERRADO PILOT
Device    : cuda
Embed dim : 8 por classe | Vocab: 128 classes
Treino: 2000 | Val: 500 | Teste: 420

Modelo    : MLPEmbedding | Parametros: 51,874
Input dim : 39 x 8 (emb) + 3 (feat) = 315
Arquitetura: 315 -> 128 -> 64 -> 32 -> 2

 Epoch    Loss Tr   BalAcc Tr   Loss Val   BalAcc Val   F1 Val
─────────────────────────────────────────────────────────────────
     1     0.6850      0.5585     0.6550       0.6420   0.4900
     2     0.6535      0.6265     0.5634       0.8040   0.7958
     3     0.6354      0.6465     0.5153       0.8180   0.8084
     4     0.6180      0.6635     0.5073       0.7900   0.7722
     5     0.6122      0.6675     0.5381       0.8000   0.7890
     6     0.6073      0.6710     0.5188       0.8000   0.7951
     7     0.5996      0.6805     0.5088       0.7960   0.7927
     8     0.6010      0.6685     0.5181       0.8120   0.8017
     9     0.5931      0.6880     0.5177       0.7940   0.7911
    10     0.5905     

In [22]:
r"""
lstm_serie.py
=============
LSTM sobre a serie temporal de classes LULC.

PRE-REQUISITO apos reiniciar o kernel:
  1. Rodar lulc_dataset.py  (define LULCTransitionDataset)
  2. Rodar este script

Diferenca em relacao ao mlp_embedding.py:
  - A serie NAO e achatada — e processada passo a passo em ordem temporal
  - O LSTM mantem memoria do que viu antes em cada passo
  - O hidden state do ultimo passo representa a trajetoria completa do pixel
  - A ordem dos anos importa: N->N->P->S e diferente de P->N->N->S

Arquitetura:
  serie (39 inteiros) -> Embedding(128, EMBED_DIM)  -> (39, EMBED_DIM)
                      -> LSTM(EMBED_DIM, HIDDEN_DIM) -> hidden state final
  concatenar com features auxiliares (3)
  -> MLP 2 camadas -> 2 classes

Hipotese: se LSTM superar MLP embedding (bal_acc=0.8222),
a ORDEM temporal da serie contem informacao alem da composicao de classes.
"""

import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.metrics import balanced_accuracy_score, f1_score, confusion_matrix
import json

# LULCTransitionDataset ja esta na memoria do kernel (rodar lulc_dataset.py antes)

INDEX_DIR = r"D:\Projetos\Cerrado\GeoFM_sampling"
SEP = "=" * 70

# ============================================================
# CONFIGURACAO
# ============================================================
BATCH_SIZE  = 32
LR          = 1e-3
EPOCHS      = 40
EMBED_DIM   = 8      # mesmo do mlp_embedding para comparacao justa
HIDDEN_DIM  = 64     # dimensao do hidden state do LSTM
N_LAYERS    = 1      # numero de camadas LSTM empilhadas
DROPOUT_MLP = 0.3
SEED        = 42
VOCAB_SIZE  = 128
SERIE_LEN   = 39
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(SEED)
np.random.seed(SEED)

# ============================================================
# 1. CARREGAR DADOS
# ============================================================
print(f"\n{SEP}")
print("LSTM SERIE TEMPORAL — CERRADO PILOT")
print(SEP)
print(f"Device     : {DEVICE}")
print(f"Embed dim  : {EMBED_DIM} | LSTM hidden: {HIDDEN_DIM} | Layers: {N_LAYERS}")

df_treino = pd.read_csv(os.path.join(INDEX_DIR, "indice_treino.csv"))
df_val    = pd.read_csv(os.path.join(INDEX_DIR, "indice_val.csv"))
df_teste  = pd.read_csv(os.path.join(INDEX_DIR, "indice_teste.csv"))

ds_treino = LULCTransitionDataset(df_treino)
ds_val    = LULCTransitionDataset(df_val)
ds_teste  = LULCTransitionDataset(df_teste)

loader_treino = DataLoader(ds_treino, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
loader_val    = DataLoader(ds_val,    batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
loader_teste  = DataLoader(ds_teste,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Treino: {len(ds_treino)} | Val: {len(ds_val)} | Teste: {len(ds_teste)}")

# ============================================================
# 2. MODELO
# ============================================================

class LSTMSerie(nn.Module):
    """
    LSTM sobre serie temporal de classes LULC com embedding.

    Fluxo detalhado:
      serie (B, 39) inteiros
        -> Embedding           -> (B, 39, EMBED_DIM)
        -> permute             -> (39, B, EMBED_DIM)   [formato esperado pelo LSTM]
        -> LSTM                -> output (39, B, HIDDEN_DIM)
                                  hidden (N_LAYERS, B, HIDDEN_DIM)
        -> hidden state final  -> (B, HIDDEN_DIM)      [resumo da trajetoria]
        -> cat com features    -> (B, HIDDEN_DIM + 3)
        -> MLP                 -> (B, 2)

    Por que usar o hidden state final e nao a media?
      O hidden state final e o que o LSTM "lembra" apos processar
      toda a sequencia — e o resumo da trajetoria com atencao ao passado.
      A media perderia a nocao de qual foi o ultimo estado.
    """
    def __init__(
        self,
        vocab_size=VOCAB_SIZE,
        embed_dim=EMBED_DIM,
        hidden_dim=HIDDEN_DIM,
        n_layers=N_LAYERS,
        n_features=3,
        dropout_mlp=DROPOUT_MLP,
    ):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)

        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=n_layers,
            batch_first=False,   # espera (seq, batch, features)
            dropout=0.0,         # dropout no LSTM so faz sentido com n_layers > 1
        )

        # MLP sobre hidden state final + features auxiliares
        mlp_input = hidden_dim + n_features
        self.classifier = nn.Sequential(
            nn.Linear(mlp_input, 64),
            nn.ReLU(),
            nn.Dropout(dropout_mlp),
            nn.Linear(64, 2),
        )

    def forward(self, serie, features, serie_len=None):
        # serie    : (B, 39) inteiros
        # features : (B, 3)
        # serie_len: (B,) comprimentos reais — usado para ignorar padding no LSTM

        B = serie.size(0)

        # embedding
        emb = self.embedding(serie.long())   # (B, 39, EMBED_DIM)
        emb = emb.permute(1, 0, 2)           # (39, B, EMBED_DIM)

        # LSTM com pack_padded_sequence para ignorar o padding a esquerda
        if serie_len is not None:
            # pack: informa ao LSTM ate onde cada sequencia e real
            packed = nn.utils.rnn.pack_padded_sequence(
                emb, serie_len.cpu(), enforce_sorted=False
            )
            _, (hidden, _) = self.lstm(packed)
        else:
            _, (hidden, _) = self.lstm(emb)

        # hidden: (N_LAYERS, B, HIDDEN_DIM) — pega a ultima camada
        h = hidden[-1]   # (B, HIDDEN_DIM)

        # concatenar com features e classificar
        x = torch.cat([h, features], dim=1)   # (B, HIDDEN_DIM + 3)
        return self.classifier(x)              # (B, 2)


model = MLPSerie_ref = None   # limpa referencia anterior se existir
model = LSTMSerie().to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())
print(f"\nModelo     : LSTMSerie | Parametros: {total_params:,}")
print(f"LSTM input : {EMBED_DIM} (embed) -> hidden {HIDDEN_DIM}")
print(f"Classifier : {HIDDEN_DIM + 3} -> 64 -> 2")

# ============================================================
# 3. TREINAMENTO
# ============================================================

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=6
)

def avaliar(loader):
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0.0
    with torch.no_grad():
        for batch in loader:
            serie     = batch["serie"].to(DEVICE)
            serie_len = batch["serie_len"]          # fica na CPU para pack
            features  = batch["features"].to(DEVICE)
            labels    = batch["label"].to(DEVICE)

            logits = model(serie, features, serie_len)
            loss   = criterion(logits, labels)
            preds  = logits.argmax(dim=1)

            total_loss  += loss.item() * len(labels)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    bal_acc  = balanced_accuracy_score(all_labels, all_preds)
    f1       = f1_score(all_labels, all_preds, zero_division=0)
    return avg_loss, bal_acc, f1


print(f"\n{'Epoch':>6}  {'Loss Tr':>9}  {'BalAcc Tr':>10}  {'Loss Val':>9}  {'BalAcc Val':>11}  {'F1 Val':>7}")
print("─" * 65)

melhor_bal_acc_val = 0.0
melhor_epoch       = 0
melhor_state       = None
historico          = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    tr_loss = 0.0
    tr_preds, tr_labels = [], []

    for batch in loader_treino:
        serie     = batch["serie"].to(DEVICE)
        serie_len = batch["serie_len"]
        features  = batch["features"].to(DEVICE)
        labels    = batch["label"].to(DEVICE)

        optimizer.zero_grad()
        logits = model(serie, features, serie_len)
        loss   = criterion(logits, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # estabiliza LSTM
        optimizer.step()

        preds = logits.argmax(dim=1)
        tr_loss += loss.item() * len(labels)
        tr_preds.extend(preds.cpu().numpy())
        tr_labels.extend(labels.cpu().numpy())

    tr_loss   /= len(loader_treino.dataset)
    tr_bal_acc = balanced_accuracy_score(tr_labels, tr_preds)

    val_loss, val_bal_acc, val_f1 = avaliar(loader_val)
    scheduler.step(val_bal_acc)

    historico.append({
        "epoch": epoch, "tr_loss": tr_loss, "tr_bal_acc": tr_bal_acc,
        "val_loss": val_loss, "val_bal_acc": val_bal_acc, "val_f1": val_f1,
    })

    if val_bal_acc > melhor_bal_acc_val:
        melhor_bal_acc_val = val_bal_acc
        melhor_epoch       = epoch
        melhor_state       = {k: v.clone() for k, v in model.state_dict().items()}

    print(f"{epoch:>6}  {tr_loss:>9.4f}  {tr_bal_acc:>10.4f}  {val_loss:>9.4f}  {val_bal_acc:>11.4f}  {val_f1:>7.4f}")

print(f"\nMelhor validacao: epoch={melhor_epoch} | bal_acc={melhor_bal_acc_val:.4f}")

# ============================================================
# 4. AVALIACAO NO TESTE
# ============================================================
print(f"\n{SEP}")
print("AVALIACAO NO TESTE — melhor checkpoint")
print(SEP)

model.load_state_dict(melhor_state)
te_loss, te_bal_acc, te_f1 = avaliar(loader_teste)

model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for batch in loader_teste:
        logits = model(
            batch["serie"].to(DEVICE),
            batch["features"].to(DEVICE),
            batch["serie_len"],
        )
        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_labels.extend(batch["label"].numpy())

cm = confusion_matrix(all_labels, all_preds)
print(f"  Balanced accuracy : {te_bal_acc:.4f}")
print(f"  F1 (positivo)     : {te_f1:.4f}")
print(f"\n  Matriz de confusao:")
print(f"              pred_0  pred_1")
print(f"    real_0  :  {cm[0,0]:5d}   {cm[0,1]:5d}")
print(f"    real_1  :  {cm[1,0]:5d}   {cm[1,1]:5d}")

# ============================================================
# 5. COMPARACAO COMPLETA
# ============================================================
print(f"\n{SEP}")
print("COMPARACAO COMPLETA")
print(SEP)

with open(os.path.join(INDEX_DIR, "baseline_results.json")) as f:
    bl = json.load(f)

bl_persistence = bl["baselines"]["persistence"]["balanced_accuracy"]
bl_logreg      = bl["baselines"]["logreg_3features"]["balanced_accuracy_test"]
mlp_bruto      = 0.5634
mlp_embed      = 0.8222

print(f"\n  {'Modelo':<40} {'Bal. Acc (teste)':>16}  {'F1 (pos)':>10}")
print(f"  {'─'*40} {'─'*16}  {'─'*10}")
print(f"  {'Persistence':<40} {bl_persistence:>16.4f}  {'—':>10}")
print(f"  {'LogReg (3 features)':<40} {bl_logreg:>16.4f}  {'—':>10}")
print(f"  {'MLP serie (valores brutos)':<40} {mlp_bruto:>16.4f}  {'0.2634':>10}")
print(f"  {'MLP embedding':<40} {mlp_embed:>16.4f}  {'0.7850':>10}")
print(f"  {'LSTM embedding (este modelo)':<40} {te_bal_acc:>16.4f}  {te_f1:>10.4f}")

delta_vs_embed = te_bal_acc - mlp_embed
print(f"\n  Ganho do LSTM sobre MLP embedding: {delta_vs_embed:+.4f}")

if te_bal_acc > mlp_embed:
    print("""
  A ORDEM temporal da serie contem informacao adicional.
  O LSTM capturou padroes sequenciais alem da composicao de classes.
  Resultado: tres modelos com progressao clara — pronto para publicar.
    """)
elif te_bal_acc > bl_persistence:
    print("""
  O LSTM nao superou o MLP embedding de forma significativa.
  Interpretacao: a COMPOSICAO de classes importa mais que a ORDEM.
  O MLP embedding ja capturou o essencial da trajetoria.
  Isso e um resultado cientifico valido — nao uma falha.
    """)
else:
    print("""
  Resultado abaixo do esperado — investigar antes de continuar.
    """)

# salvar checkpoint e historico
checkpoint_path = os.path.join(INDEX_DIR, "lstm_melhor_checkpoint.pt")
torch.save({
    "model_state": melhor_state,
    "config": {
        "embed_dim" : EMBED_DIM,
        "hidden_dim": HIDDEN_DIM,
        "n_layers"  : N_LAYERS,
        "vocab_size": VOCAB_SIZE,
        "serie_len" : SERIE_LEN,
    },
    "resultado": {
        "melhor_epoch"    : melhor_epoch,
        "val_bal_acc"     : melhor_bal_acc_val,
        "teste_bal_acc"   : te_bal_acc,
        "teste_f1"        : te_f1,
    }
}, checkpoint_path)

pd.DataFrame(historico).to_csv(
    os.path.join(INDEX_DIR, "lstm_historico.csv"), index=False
)
print(f"  Checkpoint salvo em : lstm_melhor_checkpoint.pt")
print(f"  Historico salvo em  : lstm_historico.csv")


LSTM SERIE TEMPORAL — CERRADO PILOT
Device     : cuda
Embed dim  : 8 | LSTM hidden: 64 | Layers: 1
Treino: 2000 | Val: 500 | Teste: 420

Modelo     : LSTMSerie | Parametros: 24,450
LSTM input : 8 (embed) -> hidden 64
Classifier : 67 -> 64 -> 2

 Epoch    Loss Tr   BalAcc Tr   Loss Val   BalAcc Val   F1 Val
─────────────────────────────────────────────────────────────────
     1     0.6919      0.5355     0.6951       0.4860   0.4009
     2     0.6827      0.5665     0.6894       0.5020   0.4249
     3     0.6628      0.6030     0.5738       0.7400   0.7481
     4     0.6400      0.6325     0.5421       0.7180   0.7174
     5     0.6416      0.6250     0.5503       0.7000   0.7115
     6     0.6391      0.6255     0.5443       0.7840   0.7259
     7     0.6392      0.6335     0.5717       0.6760   0.7086
     8     0.6197      0.6545     0.5057       0.7720   0.7397
     9     0.6228      0.6490     0.5246       0.7840   0.7692
    10     0.6318      0.6500     0.5615       0.6960   0.

In [26]:
r"""
stability_check.py
==================
Verifica a estabilidade do MLP embedding rodando 3 vezes
com seeds diferentes — mesmo pipeline, mesma arquitetura.

PRE-REQUISITO apos reiniciar o kernel:
  1. Rodar lulc_dataset.py  (define LULCTransitionDataset)
  2. Rodar este script

Para cada seed o script:
  1. Gera um novo indice (novos pixels sorteados)
  2. Treina o MLP embedding do zero
  3. Avalia no teste
  4. Ao final reporta media e desvio padrao entre os 3 runs

Se desvio padrao < 0.03 -> resultado estavel -> proximos passos confiaveis
Se desvio padrao >= 0.03 -> resultado fragil -> aumentar amostras primeiro
"""

import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.metrics import balanced_accuracy_score, f1_score
import json

# LULCTransitionDataset ja esta na memoria do kernel

INDEX_DIR      = r"D:\Projetos\Cerrado\GeoFM_sampling"
RASTER_CLASSES = r"D:\Projetos\Cerrado\LULC\soy2024_origin_classes_N4_P15_S39_k4_mask.tif"
DATA_DIR       = r"D:\Projetos\Cerrado\LULC"
PATTERN        = "brazil_coverage_{ano}_Cerrado.tif"
YEARS          = list(range(1985, 2025))

SEP = "=" * 70

# ============================================================
# CONFIGURACAO
# ============================================================
SEEDS       = [42, 123, 7]    # 3 seeds — indice + treino
N_TREINO    = 2000
N_VAL       = 500
N_TESTE     = 500

BATCH_SIZE  = 32
LR          = 1e-3
EPOCHS      = 30
EMBED_DIM   = 8
HIDDEN_DIMS = [128, 64, 32]
DROPOUT     = 0.3
VOCAB_SIZE  = 128
SERIE_LEN   = 39

CLASSE_POS  = 1
CLASSE_NEG  = 2
BLOCK       = 2048

FRAC_TREINO = 0.70
FRAC_VAL    = 0.15

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ============================================================
# MODELO (identico ao mlp_embedding.py)
# ============================================================

class MLPEmbedding(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(VOCAB_SIZE, EMBED_DIM, padding_idx=0)
        input_dim = SERIE_LEN * EMBED_DIM + 3
        layers, prev = [], input_dim
        for h in HIDDEN_DIMS:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(DROPOUT)]
            prev = h
        layers.append(nn.Linear(prev, 2))
        self.net = nn.Sequential(*layers)

    def forward(self, serie, features):
        emb      = self.embedding(serie.long())        # (B, 39, 8)
        emb_flat = emb.view(emb.size(0), -1)           # (B, 312)
        return self.net(torch.cat([emb_flat, features], dim=1))

# ============================================================
# FUNCOES AUXILIARES
# ============================================================

import rasterio
from rasterio.windows import Window

def _ler_pixel_local(year, row, col):
    path = os.path.join(DATA_DIR, PATTERN.format(ano=year))
    with rasterio.open(path) as ds:
        v = ds.read(1, window=Window(col, row, 1, 1), out_dtype="uint8")[0, 0]
    return int(v)

def encontrar_T(row, col):
    prev = None
    for y in YEARS:
        v = _ler_pixel_local(y, row, col)
        if v in (0, 21):
            prev = None
            continue
        if prev == 4 and v == 15:
            return y
        prev = v
    return None

def coletar_coords():
    """Varre o raster uma vez e retorna coords por classe e bloco."""
    with rasterio.open(RASTER_CLASSES) as ds:
        H, W = ds.height, ds.width

    row_treino_end = int(H * FRAC_TREINO)
    row_val_end    = int(H * (FRAC_TREINO + FRAC_VAL))

    def bloco(row):
        if row < row_treino_end: return "treino"
        if row < row_val_end:    return "val"
        return "teste"

    coords = {
        "treino": {CLASSE_POS: [], CLASSE_NEG: []},
        "val"   : {CLASSE_POS: [], CLASSE_NEG: []},
        "teste" : {CLASSE_POS: [], CLASSE_NEG: []},
    }

    print("  Varrendo raster para coletar coordenadas...")
    with rasterio.open(RASTER_CLASSES) as ds:
        for row0 in range(0, H, BLOCK):
            bh = min(BLOCK, H - row0)
            for col0 in range(0, W, BLOCK):
                bw = min(BLOCK, W - col0)
                arr = ds.read(1, window=Window(col0, row0, bw, bh))
                for cls in (CLASSE_POS, CLASSE_NEG):
                    rs, cs = np.where(arr == cls)
                    for r, c in zip(rs, cs):
                        b = bloco(row0 + r)
                        coords[b][cls].append((row0 + r, col0 + c))
    print(f"  Coletado: treino pos={len(coords['treino'][CLASSE_POS]):,} neg={len(coords['treino'][CLASSE_NEG]):,} | "
          f"val pos={len(coords['val'][CLASSE_POS]):,} | teste pos={len(coords['teste'][CLASSE_POS]):,}")
    return coords

def amostrar_indice(coords, n_total, seed):
    """Amostra pixels e encontra T para cada um."""
    rng = np.random.default_rng(seed)
    splits = [("treino", n_total), ("val", N_VAL), ("teste", N_TESTE)]
    dfs = {}
    for bloco, n in splits:
        n_cada = n // 2
        registros = []
        for cls, label in [(CLASSE_POS, 1), (CLASSE_NEG, 0)]:
            lista = coords[bloco][cls]
            idx   = rng.permutation(len(lista))[:min(len(lista), n_cada * 10)]
            for i in idx:
                if len([r for r in registros if r["label"] == label]) >= n_cada:
                    break
                row, col = lista[i]
                T = encontrar_T(row, col)
                if T is None:
                    continue
                registros.append({"row": row, "col": col, "T": T, "label": label})
        df = pd.DataFrame(registros).sample(frac=1, random_state=seed).reset_index(drop=True)
        dfs[bloco] = df
    return dfs["treino"], dfs["val"], dfs["teste"]

def treinar_e_avaliar(df_tr, df_va, df_te, seed):
    """Treina MLP embedding e retorna metricas no teste."""
    torch.manual_seed(seed)
    np.random.seed(seed)

    ds_tr = LULCTransitionDataset(df_tr)
    ds_va = LULCTransitionDataset(df_va)
    ds_te = LULCTransitionDataset(df_te)

    ld_tr = DataLoader(ds_tr, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
    ld_va = DataLoader(ds_va, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    ld_te = DataLoader(ds_te, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    model     = MLPEmbedding().to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=0.5, patience=5
    )

    melhor_val  = 0.0
    melhor_state = None

    for epoch in range(1, EPOCHS + 1):
        model.train()
        for batch in ld_tr:
            serie    = batch["serie"].to(DEVICE)
            features = batch["features"].to(DEVICE)
            labels   = batch["label"].to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(serie, features), labels)
            loss.backward()
            optimizer.step()

        # validacao
        model.eval()
        preds_va, labels_va = [], []
        with torch.no_grad():
            for batch in ld_va:
                logits = model(batch["serie"].to(DEVICE), batch["features"].to(DEVICE))
                preds_va.extend(logits.argmax(1).cpu().numpy())
                labels_va.extend(batch["label"].numpy())
        val_acc = balanced_accuracy_score(labels_va, preds_va)
        scheduler.step(val_acc)

        if val_acc > melhor_val:
            melhor_val   = val_acc
            melhor_state = {k: v.clone() for k, v in model.state_dict().items()}

    # teste com melhor checkpoint
    model.load_state_dict(melhor_state)
    model.eval()
    preds_te, labels_te = [], []
    with torch.no_grad():
        for batch in ld_te:
            logits = model(batch["serie"].to(DEVICE), batch["features"].to(DEVICE))
            preds_te.extend(logits.argmax(1).cpu().numpy())
            labels_te.extend(batch["label"].numpy())

    te_bal_acc = balanced_accuracy_score(labels_te, preds_te)
    te_f1      = f1_score(labels_te, preds_te, zero_division=0)
    n_pos_te   = int(pd.Series(labels_te).sum())

    return {
        "val_bal_acc" : round(melhor_val, 4),
        "te_bal_acc"  : round(te_bal_acc, 4),
        "te_f1"       : round(te_f1, 4),
        "n_treino"    : len(df_tr),
        "n_teste"     : len(df_te),
        "n_pos_teste" : n_pos_te,
    }

# ============================================================
# MAIN — 3 RUNS
# ============================================================
print(f"\n{SEP}")
print("STABILITY CHECK — MLP EMBEDDING (3 SEEDS)")
print(SEP)
print(f"Device: {DEVICE} | Seeds: {SEEDS} | Epochs por run: {EPOCHS}")

# varrer o raster uma unica vez (reutilizado em todos os runs)
print(f"\n[Passo 1/2] Coletando coordenadas do raster (feito uma vez)...")
coords = coletar_coords()

# 3 runs
print(f"\n[Passo 2/2] Rodando {len(SEEDS)} experimentos...\n")
resultados = []

for i, seed in enumerate(SEEDS):
    print(f"{SEP}")
    print(f"RUN {i+1}/{len(SEEDS)} — seed={seed}")
    print(f"{SEP}")

    print(f"  Gerando indice...")
    df_tr, df_va, df_te = amostrar_indice(coords, N_TREINO, seed)
    print(f"  Treino={len(df_tr)} | Val={len(df_va)} | Teste={len(df_te)} "
          f"(pos={df_te['label'].sum()}, {100*df_te['label'].mean():.1f}%)")

    print(f"  Treinando MLP embedding ({EPOCHS} epochs)...")
    res = treinar_e_avaliar(df_tr, df_va, df_te, seed)
    res["seed"] = seed
    resultados.append(res)

    print(f"  Val bal_acc : {res['val_bal_acc']:.4f}")
    print(f"  Teste bal_acc: {res['te_bal_acc']:.4f} | F1: {res['te_f1']:.4f}")

# ============================================================
# RESUMO FINAL
# ============================================================
print(f"\n{SEP}")
print("RESUMO DE ESTABILIDADE")
print(SEP)

df_res = pd.DataFrame(resultados)
te_accs = df_res["te_bal_acc"].values
te_f1s  = df_res["te_f1"].values

print(f"\n  {'Seed':>6}  {'N treino':>9}  {'N teste':>8}  {'Pos teste':>10}  {'Val BalAcc':>11}  {'Teste BalAcc':>13}  {'F1 (pos)':>9}")
print(f"  {'─'*6}  {'─'*9}  {'─'*8}  {'─'*10}  {'─'*11}  {'─'*13}  {'─'*9}")
for _, r in df_res.iterrows():
    print(f"  {int(r.seed):>6}  {int(r.n_treino):>9}  {int(r.n_teste):>8}  "
          f"{int(r.n_pos_teste):>10}  {r.val_bal_acc:>11.4f}  {r.te_bal_acc:>13.4f}  {r.te_f1:>9.4f}")

print(f"\n  {'─'*80}")
print(f"  Media  bal_acc teste : {np.mean(te_accs):.4f}")
print(f"  Desvio padrao        : {np.std(te_accs):.4f}")
print(f"  Min / Max            : {np.min(te_accs):.4f} / {np.max(te_accs):.4f}")

print(f"\n  Media  F1 teste      : {np.mean(te_f1s):.4f}")
print(f"  Desvio padrao F1     : {np.std(te_f1s):.4f}")

std = np.std(te_accs)
print(f"\n{SEP}")
if std < 0.03:
    print(f"RESULTADO ESTAVEL (std={std:.4f} < 0.03)")
    print("""
  O resultado e consistente entre diferentes amostras.
  Proximo passo recomendado: aumentar amostras (10k-20k)
  para verificar se a performance ainda melhora com mais dados.
    """)
else:
    print(f"RESULTADO FRAGIL (std={std:.4f} >= 0.03)")
    print("""
  O resultado varia significativamente entre amostras.
  Proximo passo recomendado: aumentar amostras (10k+) antes
  de qualquer outra mudanca de arquitetura ou pipeline.
    """)

# salvar
out_path = os.path.join(INDEX_DIR, "stability_results.json")
with open(out_path, "w") as f:
    json.dump({
        "seeds"   : SEEDS,
        "runs"    : resultados,
        "summary" : {
            "mean_te_bal_acc": round(float(np.mean(te_accs)), 4),
            "std_te_bal_acc" : round(float(np.std(te_accs)),  4),
            "min_te_bal_acc" : round(float(np.min(te_accs)),  4),
            "max_te_bal_acc" : round(float(np.max(te_accs)),  4),
            "mean_te_f1"     : round(float(np.mean(te_f1s)),  4),
            "std_te_f1"      : round(float(np.std(te_f1s)),   4),
            "stable"         : bool(std < 0.03),
        }
    }, f, indent=2)

print(f"Resultados salvos em: {out_path}")
print(SEP)


STABILITY CHECK — MLP EMBEDDING (3 SEEDS)
Device: cuda | Seeds: [42, 123, 7] | Epochs por run: 30

[Passo 1/2] Coletando coordenadas do raster (feito uma vez)...
  Varrendo raster para coletar coordenadas...
  Coletado: treino pos=527,553 neg=360,854 | val pos=4,805 | teste pos=170

[Passo 2/2] Rodando 3 experimentos...

RUN 1/3 — seed=42
  Gerando indice...
  Treino=2000 | Val=500 | Teste=420 (pos=170, 40.5%)
  Treinando MLP embedding (30 epochs)...
  Val bal_acc : 0.8180
  Teste bal_acc: 0.8222 | F1: 0.7850
RUN 2/3 — seed=123
  Gerando indice...
  Treino=2000 | Val=500 | Teste=420 (pos=170, 40.5%)
  Treinando MLP embedding (30 epochs)...
  Val bal_acc : 0.7660
  Teste bal_acc: 0.8222 | F1: 0.7850
RUN 3/3 — seed=7
  Gerando indice...
  Treino=2000 | Val=500 | Teste=420 (pos=170, 40.5%)
  Treinando MLP embedding (30 epochs)...
  Val bal_acc : 0.8100
  Teste bal_acc: 0.5378 | F1: 0.2069

RESUMO DE ESTABILIDADE

    Seed   N treino   N teste   Pos teste   Val BalAcc   Teste BalAcc   F1 

In [27]:
import pandas as pd
import os

INDEX_DIR = r"D:\Projetos\Cerrado\GeoFM_sampling"

# o stability_check nao salvou os indices — precisamos re-gerar so as estatisticas
# mas podemos comparar com o indice original salvo
df_orig = pd.read_csv(os.path.join(INDEX_DIR, "indice_treino.csv"))

print("Indice original (seed=42):")
print(f"  T min/max por classe:")
print(df_orig.groupby('label')['T'].describe()[['min','50%','max']])

Indice original (seed=42):
  T min/max por classe:
          min     50%     max
label                        
0      1986.0  2012.0  2019.0
1      1986.0  2014.0  2020.0


In [28]:
r"""
gerar_indice_raster.py
======================
Gera indice de pixels com split espacial e amostragem estratificada por T.

Problema identificado no stability check:
  Com 1.000 positivos em 35 anos de T (1986-2020), alguns seeds deixam
  certos anos sem representacao no treino — o modelo nao generaliza.

Solucao:
  Amostragem estratificada por T — cada ano recebe uma quota minima
  de pixels. Isso garante cobertura da distribuicao de T independente
  do seed de amostragem.

Split espacial (mantido):
  Treino : 70% das linhas (norte/centro)
  Val    : 15% das linhas (intermediario)
  Teste  : 15% das linhas (sul)
"""

import os
import numpy as np
import pandas as pd
import rasterio
from rasterio.windows import Window

# ============================================================
# CONFIGURACAO
# ============================================================
RASTER_CLASSES = r"D:\Projetos\Cerrado\LULC\soy2024_origin_classes_N4_P15_S39_k4_mask.tif"
DATA_DIR       = r"D:\Projetos\Cerrado\LULC"
PATTERN        = "brazil_coverage_{ano}_Cerrado.tif"
YEARS          = list(range(1985, 2025))
OUT_DIR        = r"D:\Projetos\Cerrado\GeoFM_sampling"

CLASSE_POS = 1
CLASSE_NEG = 2

# Amostras por split — aumentado para 10k no treino
N_TREINO = 10000   # 5000 pos + 5000 neg
N_VAL    =  1000   #  500 pos +  500 neg
N_TESTE  =   500   #  250 pos +  250 neg

# Amostragem estratificada por T no treino
# Cada ano de T recebe pelo menos QUOTA_MIN_POR_ANO amostras por classe
QUOTA_MIN_POR_ANO = 20   # 20 pos + 20 neg por ano = cobertura minima

BLOCK       = 2048
SEED        = 42
FRAC_TREINO = 0.70
FRAC_VAL    = 0.15

SEP = "=" * 70

# ============================================================
# AUXILIARES
# ============================================================

def _path_lulc(year):
    return os.path.join(DATA_DIR, PATTERN.format(ano=year))

def _ler_pixel(year, row, col):
    with rasterio.open(_path_lulc(year)) as ds:
        v = ds.read(1, window=Window(col, row, 1, 1), out_dtype="uint8")[0, 0]
    return int(v)

def encontrar_T(row, col):
    prev = None
    for y in YEARS:
        v = _ler_pixel(y, row, col)
        if v in (0, 21):
            prev = None
            continue
        if prev == 4 and v == 15:
            return y
        prev = v
    return None

# ============================================================
# PASSO 1: COLETAR COORDENADAS
# ============================================================
print(f"\n{SEP}")
print("PASSO 1: Coletando coordenadas por classe e bloco espacial")
print(SEP)

with rasterio.open(RASTER_CLASSES) as ds:
    H, W = ds.height, ds.width

row_treino_end = int(H * FRAC_TREINO)
row_val_end    = int(H * (FRAC_TREINO + FRAC_VAL))

def bloco_de(row):
    if row < row_treino_end: return "treino"
    if row < row_val_end:    return "val"
    return "teste"

coords = {
    "treino": {CLASSE_POS: [], CLASSE_NEG: []},
    "val"   : {CLASSE_POS: [], CLASSE_NEG: []},
    "teste" : {CLASSE_POS: [], CLASSE_NEG: []},
}

with rasterio.open(RASTER_CLASSES) as ds:
    for row0 in range(0, H, BLOCK):
        bh = min(BLOCK, H - row0)
        for col0 in range(0, W, BLOCK):
            bw = min(BLOCK, W - col0)
            arr = ds.read(1, window=Window(col0, row0, bw, bh))
            for cls in (CLASSE_POS, CLASSE_NEG):
                rs, cs = np.where(arr == cls)
                for r, c in zip(rs, cs):
                    b = bloco_de(row0 + r)
                    coords[b][cls].append((row0 + r, col0 + c))

        if row0 % (BLOCK * 8) == 0:
            print(f"  Linha {row0:6d}/{H} | "
                  f"treino pos={len(coords['treino'][CLASSE_POS]):,} "
                  f"neg={len(coords['treino'][CLASSE_NEG]):,}")

print(f"\nCoordenadas coletadas:")
for b in ("treino", "val", "teste"):
    print(f"  {b:6s}: pos={len(coords[b][CLASSE_POS]):,}  neg={len(coords[b][CLASSE_NEG]):,}")

# ============================================================
# PASSO 2: AMOSTRAGEM ESTRATIFICADA POR T (treino)
# ============================================================
print(f"\n{SEP}")
print("PASSO 2: Amostragem estratificada por T (treino)")
print(SEP)

rng = np.random.default_rng(SEED)

def amostrar_estratificado(lista_coords, n_total, label, desc, quota_min=QUOTA_MIN_POR_ANO):
    """
    Amostra pixels garantindo quota minima por ano de T.
    Estrategia:
      1. Sorteia candidatos (5x o necessario)
      2. Encontra T para cada um
      3. Para cada ano com menos que quota_min amostras, busca mais pixels
      4. Completa ate n_total com amostragem aleatoria do restante
    """
    print(f"\n  [{desc}] Buscando {n_total} pixels com quota minima de {quota_min}/ano...")

    # fase 1: candidatos iniciais
    idx        = rng.permutation(len(lista_coords))[:min(len(lista_coords), n_total * 8)]
    candidatos = [lista_coords[i] for i in idx]

    por_ano    = {}   # ano -> lista de registros
    verificados = 0

    for row, col in candidatos:
        if sum(len(v) for v in por_ano.values()) >= n_total * 2:
            break
        T = encontrar_T(row, col)
        if T is None:
            continue
        verificados += 1
        if T not in por_ano:
            por_ano[T] = []
        por_ano[T].append({"row": row, "col": col, "T": T, "label": label})

        if verificados % 500 == 0:
            total = sum(len(v) for v in por_ano.values())
            anos_ok = sum(1 for v in por_ano.values() if len(v) >= quota_min)
            print(f"    verificados={verificados} | pixels_com_T={total} | "
                  f"anos_com_quota={anos_ok}/{len(por_ano)}")

    # fase 2: selecionar com quota minima garantida
    registros = []
    # primeiro: quota minima de cada ano
    for ano in sorted(por_ano.keys()):
        amostra = por_ano[ano][:quota_min]
        registros.extend(amostra)

    # depois: completar aleatoriamente ate n_total
    restantes = []
    for ano, lista in por_ano.items():
        restantes.extend(lista[quota_min:])
    rng.shuffle(restantes)

    faltam = n_total - len(registros)
    if faltam > 0:
        registros.extend(restantes[:faltam])

    anos_cobertos  = len(por_ano)
    anos_com_quota = sum(1 for v in por_ano.values() if len(v) >= quota_min)
    print(f"    Total: {len(registros)} | Anos cobertos: {anos_cobertos} | "
          f"Anos com quota >= {quota_min}: {anos_com_quota}")
    return registros[:n_total]

def amostrar_simples(lista_coords, n_total, label, desc, max_factor=10):
    """Amostragem simples para val e teste."""
    idx       = rng.permutation(len(lista_coords))[:min(len(lista_coords), n_total * max_factor)]
    registros = []
    print(f"\n  [{desc}] Buscando {n_total} pixels...")
    for i in idx:
        if len(registros) >= n_total:
            break
        row, col = lista_coords[i]
        T = encontrar_T(row, col)
        if T is None:
            continue
        registros.append({"row": row, "col": col, "T": T, "label": label})
        if len(registros) % 100 == 0:
            print(f"    {len(registros)}/{n_total}")
    print(f"    Encontrados: {len(registros)}")
    return registros

# treino — estratificado
reg_tr_pos = amostrar_estratificado(coords["treino"][CLASSE_POS], N_TREINO // 2, 1, "treino pos")
reg_tr_neg = amostrar_estratificado(coords["treino"][CLASSE_NEG], N_TREINO // 2, 0, "treino neg")

# val e teste — simples
print(f"\n{SEP}")
print("PASSO 3: Amostragem simples para val e teste")
print(SEP)

reg_va_pos = amostrar_simples(coords["val"][CLASSE_POS],   N_VAL   // 2, 1, "val pos")
reg_va_neg = amostrar_simples(coords["val"][CLASSE_NEG],   N_VAL   // 2, 0, "val neg")
reg_te_pos = amostrar_simples(coords["teste"][CLASSE_POS], N_TESTE // 2, 1, "teste pos")
reg_te_neg = amostrar_simples(coords["teste"][CLASSE_NEG], N_TESTE // 2, 0, "teste neg")

# ============================================================
# PASSO 4: SALVAR
# ============================================================
print(f"\n{SEP}")
print("PASSO 4: Salvando indices")
print(SEP)

os.makedirs(OUT_DIR, exist_ok=True)

def salvar(pos, neg, nome):
    df = pd.DataFrame(pos + neg).sample(frac=1, random_state=SEED).reset_index(drop=True)
    path = os.path.join(OUT_DIR, nome)
    df.to_csv(path, index=False)
    n_pos = int(df['label'].sum())
    print(f"\n  {nome}")
    print(f"    Total    : {len(df)}")
    print(f"    Positivos: {n_pos} ({100*n_pos/len(df):.1f}%)")
    print(f"    T min/max: {df['T'].min()}–{df['T'].max()}")
    print(f"    T por classe (mediana):")
    print(df.groupby('label')['T'].describe()[['min','50%','max']].to_string())
    print(f"    Salvo em : {path}")
    return df

df_treino = salvar(reg_tr_pos, reg_tr_neg, "indice_treino.csv")
df_val    = salvar(reg_va_pos, reg_va_neg, "indice_val.csv")
df_teste  = salvar(reg_te_pos, reg_te_neg, "indice_teste.csv")

# ============================================================
# PASSO 5: VERIFICACAO DE COBERTURA DE T
# ============================================================
print(f"\n{SEP}")
print("PASSO 5: Cobertura de T no treino (anos com >= quota pixels por classe)")
print(SEP)

for label, nome in [(1, "positivos"), (0, "negativos")]:
    sub = df_treino[df_treino['label'] == label]
    contagem = sub.groupby('T').size()
    anos_ok  = (contagem >= QUOTA_MIN_POR_ANO).sum()
    print(f"\n  {nome}: {len(sub)} amostras em {len(contagem)} anos")
    print(f"  Anos com >= {QUOTA_MIN_POR_ANO} amostras: {anos_ok}/{len(contagem)}")
    print(f"  Anos com < {QUOTA_MIN_POR_ANO} amostras : {(contagem < QUOTA_MIN_POR_ANO).sum()}")
    escassos = contagem[contagem < QUOTA_MIN_POR_ANO]
    if len(escassos) > 0:
        print(f"  Anos escassos: {escassos.to_dict()}")

print(f"\n{SEP}")
print("CONCLUIDO")
print(SEP)
print("""
Proximos passos:
  1. Rodar stability_check.py para verificar se std < 0.03
  2. Se estavel -> resultado confirmado -> documentar
  3. Se ainda fragil -> investigar distribuicao geografica do teste
""")


PASSO 1: Coletando coordenadas por classe e bloco espacial
  Linha      0/82933 | treino pos=9 neg=1
  Linha  16384/82933 | treino pos=31,112 neg=15,694
  Linha  32768/82933 | treino pos=350,780 neg=199,528
  Linha  49152/82933 | treino pos=513,640 neg=345,162
  Linha  65536/82933 | treino pos=527,553 neg=360,854
  Linha  81920/82933 | treino pos=527,553 neg=360,854

Coordenadas coletadas:
  treino: pos=527,553  neg=360,854
  val   : pos=4,805  neg=10,018
  teste : pos=170  neg=2,467

PASSO 2: Amostragem estratificada por T (treino)

  [treino pos] Buscando 5000 pixels com quota minima de 20/ano...
    verificados=500 | pixels_com_T=500 | anos_com_quota=6/33
    verificados=1000 | pixels_com_T=1000 | anos_com_quota=6/35
    verificados=1500 | pixels_com_T=1500 | anos_com_quota=7/35
    verificados=2000 | pixels_com_T=2000 | anos_com_quota=10/35
    verificados=2500 | pixels_com_T=2500 | anos_com_quota=15/35
    verificados=3000 | pixels_com_T=3000 | anos_com_quota=23/36
    verificado